In [ ]:
import sys
import os
# Use the current Jupyter kernel's Python to install the packages
#!{sys.executable} -m pip install pyomo pandas typing_extensions packaging pyvis ipython
current_dir = os.path.dirname(os.path.abspath('__file__')) 
sys.path.append(os.path.abspath('.'))

In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2
from pyomo.environ import *
from itertools import permutations
import pandas as pd
from collections import deque
from typing import List, Tuple, Dict
from types import SimpleNamespace
import sys
import sysconfig

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None


def display_dataframe_to_user(name=None, dataframe=None):
    if dataframe is None:
        return None
    if display is not None:
        if name and Markdown is not None:
            display(Markdown(f"**{name}**"))
        display(dataframe)
    else:
        if name:
            print(name)
        try:
            print(dataframe)
        except Exception:
            pass
    return dataframe


tools = SimpleNamespace(display_dataframe_to_user=display_dataframe_to_user)

# Print full Python version information
print("Python version:", sys.version)

# Check compile-time flag: was this Python built with GIL disabled?
gil_flag = sysconfig.get_config_var("Py_GIL_DISABLED")
print("Py_GIL_DISABLED config var:", gil_flag)

# Check at runtime whether the GIL is currently enabled (Python 3.13+)
if hasattr(sys, "_is_gil_enabled"):
    print("GIL enabled at runtime:", sys._is_gil_enabled())
else:
    print("sys._is_gil_enabled() is not available on this Python version")


# Optimal Operation Planning for a Modular Process

## Introduction

This notebook implements a **state-space search and optimization framework** for modular chemical processes, using BFS exploration combined with Pyomo optimization.

The following elements are defined:

* A set of **four modular units** (*ModPlant*, `HC10`–`HC40`), each with:

  * **Operations** (e.g., Filling, Draining, Stirring, Settling, Connect/Disconnect) with parameters and costs.
  * **Interfaces** (input/output ports) that allow material transfer.
  * **Capacity constraints** (maximum volume).
  * **Initial resources** (starting materials).
* A parsed ISA-88 **General Recipe** that is transformed into **semantic reaction rules** (Dosing, Mix, Usage, Settling, Separation).
* A **BFS engine** that builds the full state-transition graph:

  * Connects ModPlant,
  * Transfers and transforms materials,
  * Applies recipe rules.
* A **Pyomo model** that selects an optimal path of transitions from the initial state to the defined end state.

### Objective

The objective is to compute a **feasible and cost-efficient sequence of operations** that transforms the initial ModPlant contents into the recipe-defined end state.


---

## ModPlant Operations and Costs

The `modplant_ops` mapping lists, for each ModPlant (`HC10`–`HC40`), the operations it can perform along with an operation-specific parameter and a cost weight. Each entry is a 3-tuple:

* **Operation Type** — the action (e.g., `Filling`, `Draining`, `Stirring`, `Settling`, `Connect`, `Disconnect`, `None`).
* **Operation Parameter** — an optional value (string or number) used by the operation (left empty `""` where not applicable).
* **Cost** — a unitless per-operation weight used for planning/optimization (not per unit of material).

These costs model the relative “effort” of executing an operation and can encode time, energy, or complexity as defined by your optimization logic. Each ModPlant exposes a fixed set of operations that reflects its role in the process chain.

The code then flattens this structure into a tabular view with the columns **ModPlant**, **Operation Type**, **Operation Param**, and **Cost** for inspection or downstream use.


In [ ]:
# ---- Original definitions (kept for reproducibility) ----
modplant_ops = {
    'HC10': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Stirring', '100', 3), ('Stirring', '200', 3),('Connect', '', 1), ('Disconnect', '', 0),  ('None', '', 0)],
    'HC20': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Stirring', '150', 3), ('Stirring', '300', 3), ('Connect', '', 1), ('Disconnect', '', 0), ('None', '', 0)],
    'HC30': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Stirring', '100', 3), ('Stirring', '150', 3),('Connect', '', 1), ('Disconnect', '', 0),  ('None', '', 0)],
    'HC40': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Connect', '', 1), ('Disconnect', '', 0), ('None', '', 0)]
}

modplant_interfaces = {
    'HC10': [('Input', 'HC10_In1'), ('Input', 'HC10_In2'), ('Input', 'HC10_In3'), ('Output', 'HC10_Out1'), ('Output', 'HC10_Out2'), ('Output', 'HC10_Out3')],
    'HC20': [('Input', 'HC20_In1'), ('Input', 'HC20_In2'), ('Input', 'HC20_In3'), ('Output', 'HC20_Out1'), ('Output', 'HC20_Out2'), ('Output', 'HC20_Out3')],
    'HC30': [('Input', 'HC30_In1'), ('Input', 'HC30_In2'), ('Input', 'HC30_In3'), ('Input', 'HC30_In4'), ('Output', 'HC30_Out1')],
    'HC40': [('Input', 'HC40_In1'), ('Output', 'HC40_Out1'), ('Output', 'HC40_Out2')],
}



modplant_maximum_volume = {
    'HC10': [10],
    'HC20': [15],
    'HC30': [10],
    'HC40': [30]
}

modplant_resources = {
    'HC10': ['A', 10],
    'HC20': ['B', 10],
    'HC30': ['C', 10],
}


In [ ]:
import random
import os

# Toggle between original fixed modplants and random generation
use_original_modplants = True
env_seed = os.getenv('MODPLANT_SEED')
random_seed = int(env_seed) if env_seed is not None else random.randint(1, 10000)  # You can set this to a fixed value for reproducibility
min_modplant_count = 4
max_modplant_count = 4

# ---- Random generator under given constraints ----
rate_choices = [0.1, 0.2, 0.3]
rpm_choices = list(range(50, 301, 50))


def generate_random_modplants(seed: int, min_n: int, max_n: int):
    random.seed(seed)
    n = random.randint(max(4, min_n), min(6, max_n))
    names = [f"HC{(i+1)*10}" for i in range(n)]

    w_ops = {}
    w_ifaces = {}
    w_cap = {}

    # Ensure at least one Settling and one Stirring holder
    settling_assigned = False
    stirring_assigned = False

    for idx, name in enumerate(names):
        # interfaces
        num_inputs = random.randint(1, 4)
        num_outputs = random.randint(1, 4)
        w_ifaces[name] = [("Input", f"{name}_In{i+1}") for i in range(num_inputs)] +                           [("Output", f"{name}_Out{i+1}") for i in range(num_outputs)]

        # capacity
        max_vol = random.choice(range(10, 31, 5))
        w_cap[name] = [max_vol]

        # base ops
        drain_rate = random.choice(rate_choices)
        fill_rate = random.choice(rate_choices)
        ops = [
            ("Draining", drain_rate, 3),
            ("Filling", fill_rate, 0),
            ("Connect", "", 1),
            ("Disconnect", "", 0),
            ("None", "", 0),
        ]

        # Settling (ensure at least one)
        if not settling_assigned or random.random() < 0.5:
            ops.append(("Settling", "", random.randint(1, 3)))
            settling_assigned = True

        # Stirring 1-2 params (ensure at least one overall)
        stir_count = random.randint(1, 2)
        rpms = random.sample(rpm_choices, stir_count)
        for rpm in rpms:
            ops.append(("Stirring", str(rpm), random.randint(1, 4)))
        stirring_assigned = stirring_assigned or bool(rpms)

        w_ops[name] = ops

    # Guarantee constraints if randomness skipped them
    if not settling_assigned:
        name = names[0]
        w_ops[name].append(("Settling", "", random.randint(1, 3)))
    if not stirring_assigned:
        name = names[0]
        rpm = random.choice(rpm_choices)
        w_ops[name].append(("Stirring", str(rpm), random.randint(1, 4)))

    # resources: assign A,B,C to three distinct modplants
    resources = {}
    chosen = random.sample(names, k=min(3, len(names)))
    for mat, wb in zip(['A', 'B', 'C'], chosen):
        cap = w_cap[wb][0]
        qty = min(random.randint(5, 15), cap)
        resources[wb] = [mat, qty]

    return w_ops, w_ifaces, w_cap, resources

# ---- Choose config ----
if not use_original_modplants:
    modplant_ops, modplant_interfaces, modplant_maximum_volume, modplant_resources = generate_random_modplants(
        seed=random_seed,
        min_n=min_modplant_count,
        max_n=max_modplant_count,
    )

# === ModPlant summary table after generation ===
# Expanded per-operation rows for structured display
summary_rows = []
for wb in sorted(modplant_ops.keys()):
    ifaces = modplant_interfaces.get(wb, [])
    num_inputs = sum(1 for t, _ in ifaces if t == 'Input')
    num_outputs = sum(1 for t, _ in ifaces if t == 'Output')
    max_vols = modplant_maximum_volume.get(wb, [])
    max_vol = max_vols[0] if max_vols else None
    res = modplant_resources.get(wb)
    res_str = f"{res[0]}:{res[1]}" if res else ''

    ops = modplant_ops.get(wb, [])
    if not ops:
        summary_rows.append({
            'ModPlant': wb,
            'Inputs': num_inputs,
            'Outputs': num_outputs,
            'MaxVolume': max_vol,
            'Resources': res_str,
            'Operation Type': '',
            'Operation Param': '',
            'Operation Cost': '',
        })
    else:
        for op_type, op_param, op_cost in ops:
            summary_rows.append({
                'ModPlant': wb,
                'Inputs': num_inputs,
                'Outputs': num_outputs,
                'MaxVolume': max_vol,
                'Resources': res_str,
                'Operation Type': op_type,
                'Operation Param': op_param,
                'Operation Cost': op_cost,
            })

try:
    import pandas as pd
    df_modplant_summary = pd.DataFrame(summary_rows)
    # Display in notebook UI if available
    tools.display_dataframe_to_user(name='ModPlant Summary (Expanded)', dataframe=df_modplant_summary)
    # Also print ASCII table for batch_runner visibility
    print('ModPlant Summary (Expanded):')
    try:
        print(df_modplant_summary.to_string(index=False))
    except Exception:
        for row in summary_rows:
            print(row)
    # cleanup dataframe to reduce per-seed memory
    try:
        import gc
        del df_modplant_summary
        gc.collect()
    except Exception:
        pass
except Exception:
    print('ModPlant Summary (Expanded):')
    for row in summary_rows:
        print(row)




In [ ]:
# # Create DataFrame from modplant_ops
df_ops = pd.DataFrame([
    {"ModPlant": wb, "Operation Type": op[0], "Operation Param": op[1], "Cost": op[2]}
    for wb, op_list in modplant_ops.items()
    for op in op_list
])

# Display DataFrame
tools.display_dataframe_to_user(name="ModPlant Operations", dataframe=df_ops)

# cleanup dataframe to reduce per-seed memory
try:
    import gc
    del df_ops
    gc.collect()
except Exception:
    pass


## ModPlant I/O Interfaces

This snippet programmatically defines standardized input/output interfaces for each ModPlant and flattens them into a table for inspection.

* **Input source**: `hc_data` is a list of 3-tuples `(HC_name, num_inputs, num_outputs)`, e.g. `('HC10', 3, 3)`.


* **Generated structure**: `modplant_interfaces` becomes a dictionary
  `HC → [ ('Input' | 'Output', '<HC>_In#|Out#'), ... ]`
  with deterministic naming and ordering:

  * Inputs are created first, named `<HC>_In1 … <HC>_InN`.
  * Outputs follow, named `<HC>_Out1 … <HC>_OutM`.
  * Numbering starts at **1** and is gap-free.

* **Naming convention**: The `<HC>_<In/Out><index>` pattern guarantees uniqueness across all ModPlant and makes direction explicit, which simplifies validation and graph construction in downstream algorithms.

* **Tabular view**: The code builds a DataFrame with:

  * **ModPlant** — the ModPlant identifier (`HC10`, `HC20`, …)
  * **Interface Type** — `Input` or `Output`
  * **Interface Name** — canonical interface label (e.g., `HC30_In4`)

In [ ]:
# # Input data: (HC_name, num_inputs, num_outputs)


# #hc_data = [('HC10', 5, 5), ('HC20', 5, 5), ('HC30', 5, 5), ('HC40', 5, 5)]


# # Generate modplant_interfaces dictionary
# hc_data = [('HC10', 3, 3), ('HC20', 3, 3), ('HC30', 4, 1), ('HC40', 1, 2)]
# modplant_interfaces = {}
# for hc_name, num_inputs, num_outputs in hc_data:
#     interfaces = []
#     # Add input interfaces
#     for i in range(1, num_inputs + 1):
#         interfaces.append(('Input', f'{hc_name}_In{i}'))
#     # Add output interfaces
#     for i in range(1, num_outputs + 1):
#         interfaces.append(('Output', f'{hc_name}_Out{i}'))
#     modplant_interfaces[hc_name] = interfaces

# # # Create DataFrame from modplant_interfaces
df_interfaces = pd.DataFrame([
    {"ModPlant": wb, "Interface Type": iface[0], "Interface Name": iface[1]}
    for wb, iface_list in modplant_interfaces.items()
    for iface in iface_list
])

# Display DataFrame
tools.display_dataframe_to_user(name="ModPlant Interfaces", dataframe=df_interfaces)

# cleanup dataframe to reduce per-seed memory
try:
    import gc
    del df_interfaces
    gc.collect()
except Exception:
    pass


## ModPlant Maximum Volumes

This section defines the **capacity limits** of each ModPlant in terms of maximum processable volume.

* **Data structure**:
  `modplant_maximum_volume` is a dictionary mapping each ModPlant (`HC10`–`HC40`) to a list containing its maximum volume (e.g., `HC20 → 15`).
  Although currently defined as single-element lists, this design allows for future extensions (e.g., multiple operating modes or time-dependent capacity values).

* **Tabular representation**:
  The code expands the dictionary into a DataFrame with two columns:

  * **ModPlant** — the identifier of the ModPlant.
  * **Maximum Volume** — the defined capacity limit (unit depends on process context, e.g., liters).

In [ ]:
df_maximum_volume = pd.DataFrame([
    {"ModPlant": wb, "Maximum Volume": vol}
    for wb, vols in modplant_maximum_volume.items()
    for vol in vols
])

tools.display_dataframe_to_user(name="ModPlant Maximum Volumes", dataframe=df_maximum_volume)


# cleanup dataframe to reduce per-seed memory
try:
    import gc
    del df_maximum_volume
    gc.collect()
except Exception:
    pass


## ModPlant Initial Resources

This defines the **starting contents** of each ModPlant:

* **`modplant_resources`** maps each ModPlant (`HC10`–`HC30`) to a material label and its initial quantity.
* Flattened into a DataFrame with columns:

  * **ModPlant** — container name
  * **Material** — substance initially stored
  * **Quantity** — volume/amount at start (unit depends on process, e.g., liters)

In [ ]:
df_resources = pd.DataFrame([
    {"ModPlant": wb, "Material": res[0], "Quantity": res[1]}
    for wb, res in modplant_resources.items()
])

tools.display_dataframe_to_user(name="ModPlant Resources", dataframe=df_resources)


# cleanup dataframe to reduce per-seed memory
try:
    import gc
    del df_resources
    gc.collect()
except Exception:
    pass


---

## Initial State Construction and Display

This code builds an **immutable state snapshot** of all ModPlant, combining their initial contents and output port connections, then renders it in two tables.

* **`res_to_tuple(v)`**
  Normalizes resource data into a consistent tuple of strings:

  * Supports dicts (`{"material":…, "quantity":…, "unit":…}`)
  * Lists/tuples of dicts
  * Simple `(material, quantity)` pairs
  * Formats as `"Material : quantity unit"`

* **`build_initial_state(modplant_resources, modplant_interfaces)`**
  Creates the starting state as a sorted tuple of per-ModPlant records:

  * **Content** — current material(s) in the ModPlant
  * **Outputs** — all output interfaces initialized as unconnected (`""`)

* **`display_state(state_tuple)`**
  Generates two DataFrames:

  * **Initial ModPlant Contents** — which material and how much each ModPlant holds
  * **Initial ModPlant Ports (Connections)** — all output ports with current connections (empty at start)

* **`start_state`**
  The initial system state, used as input for BFS, optimization, or simulation routines.


In [ ]:
def res_to_tuple(v):
    """Convert resource list to sorted tuple of strings"""
    if isinstance(v, dict):
        return (f"{v.get('material','')} : {float(v.get('quantity',0)):.1f} {v.get('unit','litre')}",)
    if isinstance(v, (list, tuple)) and v and isinstance(v[0], dict):
        return tuple(sorted(
            f"{d.get('material','')} : {float(d.get('quantity',0)):.1f} {d.get('unit','litre')}"
            for d in v
        ))
    if isinstance(v, (list, tuple)) and len(v) >= 2 and isinstance(v[0], str):
        return (f"{v[0]} : {float(v[1]):.1f} litre",)
    return ()

def build_initial_state(modplant_resources, modplant_interfaces):
    state = tuple(sorted(
        (
            wb,
            ("content", res_to_tuple(modplant_resources.get(wb, []))),
            ("outputs", tuple(sorted(
                (name, "", "") for t, name in ifaces if t == "Output"  # Modified: from (name, "") to (name, "", "")
            )))
        )
        for wb, ifaces in modplant_interfaces.items()
    ))
    return state

def display_state(state_tuple):
    """Display two DataFrames from the immutable state tuple."""

    df_state_contents = pd.DataFrame([
        {
            "ModPlant": wb,
            "Content": ", ".join(sorted(content_val))
        }
        for wb, (content_key, content_val), _ in state_tuple
    ])

    df_state_ports = pd.DataFrame([
        {
            "ModPlant": wb,
            "Port Type": "Output",
            "Port Name": port_name,
            "Connected To": target or "",
            "Material": material or ""  # Added Material column
        }
        for wb, _, (out_key, out_ports) in state_tuple
        for port_name, target, material in out_ports  # Modified: from port_name, target to port_name, target, material
    ])

    tools.display_dataframe_to_user(name="Initial ModPlant Contents", dataframe=df_state_contents)
    tools.display_dataframe_to_user(name="Initial ModPlant Ports (Connections)", dataframe=df_state_ports)

start_state = build_initial_state(modplant_resources, modplant_interfaces)
display_state(start_state)

## End State Construction

This function generates a **target state** that mirrors the structure of the initial state but enforces a uniform end condition.

* **`build_end_state_from_start(start_state_tuple)`**

  * Iterates over all ModPlant from the starting state.
  * Sets **content** to a fixed marker `("End",)` for every ModPlant.
  * Resets **outputs** to the same set of ports as in the start state, all left unconnected (`""`).
  * Returns a sorted, immutable tuple representing the canonical **end state**.

* **`end_state`**
  The result of applying the function to `start_state`.
  It represents the **goal configuration**: all ModPlant are marked finished, and no active connections remain.

* **`display_state(end_state)`**
  Shows two DataFrames:

  * **ModPlant Contents** — all entries display `End`.
  * **ModPlant Ports** — same ports as the start state, but all disconnected.

In [ ]:
def build_end_state_from_start(start_state_tuple):
    """Create an end state where all ModPlant have 'End' and outputs are empty but identical."""
    end_state = tuple(sorted(
        (
            wb,
            ("content", ("End",)),
            ("outputs", tuple(sorted((port_name, "", "") for port_name, _, _ in outputs)))  # Modified here
        )
        for wb, (_, _), (_, outputs) in start_state_tuple
    ))
    return end_state

end_state = build_end_state_from_start(start_state)
display_state(end_state)

---

##  Generate General Recipe
This code cell automates the generation of an ISA-88-compliant **General Recipe XML** for modular production units ("ModPlant") based on a user-defined order. It performs the following steps:

1. **Imports Required Functions**

   * `build_schedule`: Parses a user-defined production order into an executable process flow.
   * `save_general_recipe_xml`: Converts the process flow into a General Recipe in XML format.

2. **Defines a Flexible Production Order (`order`)**

   * Specifies the **total volume**, **input material sequence**, **mixing parameters**, and other operations.
   * Materials and ratios (e.g. A:1, B:2, C:3) are user-defined.
   * Optional operations include: **Mixing**, **Usage**, **Settling**, and **Separation** — depending on the order content.

3. **Generates and Displays the Flow Schedule**

   * Builds and prints an ASCII diagram showing the exact sequence of operations based on the input order.

4. **Saves the General Recipe as XML**

   * Outputs a fully structured XML recipe file ready for use in batch automation or process simulation systems.


In [ ]:
from supporting_scripts.ModPlant_Flow_Generator import build_schedule
# from supporting_scripts.ModPlant_Render_Tools import print_flowchart
from supporting_scripts.ModPlant_Flow_To_General_Recipe import save_general_recipe_xml
import sys
import io
import random

# Force UTF-8 output encoding
if hasattr(sys.stdout, "buffer"):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

# Toggle to use the original fixed recipe
use_original_order = use_original_modplants

# Original fixed recipe
order = {
  "volume": 6.0,
  "order": [
    "A",
    "B",
    "C",
    {"mix": {"rpm": 150, "duration": 30}}
  ],
  "ratio": {
    "A": [1],
    "B": [2],
    "C": [3]
  },
  "usage_and_settling": [3600, 300]
}



if not use_original_order:
    # Randomized order builder controlled by random_seed
    random.seed(random_seed)
    # volume: fixed 10L for randomized orders
    order_volume = 10

    # letters A,B,C in random order
    letters = ['A', 'B', 'C']
    random.shuffle(letters)

    # helper to create a mix step
    rpm_choices = list(range(50, 301, 50))
    def make_mix():
        return {"mix": {"rpm": random.choice(rpm_choices), "duration": random.randrange(100, 1001, 100)}}

    # Build order list with at most one mix (placed at the end)
    order_list = letters.copy()
    # Append a single mix step to keep behavior consistent with original flow
    order_list.append(make_mix())

    # ratio integers summing to 10
    parts = [random.randint(1, 8) for _ in letters]
    scale = 10 / sum(parts)
    ratio_vals = [max(1, round(v * scale)) for v in parts]
    adj = 10 - sum(ratio_vals)
    ratio_vals[0] += adj  # fix sum exactly to 10
    ratio = {k: [v] for k, v in zip(letters, ratio_vals)}

    # usage and settling durations: multiples of 100
    usage_and_settling = [random.randrange(100, 5001, 100), random.randrange(100, 1001, 100)]

    order = {
      "volume": float(order_volume),
      "order": order_list,
      "ratio": ratio,
      "usage_and_settling": usage_and_settling
    }

# Profit factor for the order (a * volume , b)
# From the time the order is placed, every second of delay in delivery will result in a loss of b.
order_profit_factor = (50 * order["volume"], -0.5)

# Generate the schedule from the order
print("Generating schedule from order.")
schedule = build_schedule(order)


In [ ]:
from typing import List, Dict

# === Helper functions for rendering ===

def fmt_time(seconds: float) -> str:
    """Helper to format seconds into a readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds // 60:.0f}m {seconds % 60:.1f}s"

def _fmt_ratio_value(val) -> str:
    """Helper to format ratio values (handles lists or single numbers)."""
    if isinstance(val, list):
        return str(val[0])
    return str(val)

def make_box(title: str, lines: List[str]) -> List[str]:
    """Build a single ASCII box with a title + lines."""
    content = [title] + lines
    # Calculate width based on the longest line
    width = max((len(x) for x in content), default=0) + 2
    top = "┌" + "─" * width + "┐"
    bot = "└" + "─" * width + "┘"
    body = ["│ " + x.ljust(width - 2) + " │" for x in content]
    return [top] + body + [bot]

def _render_box_from_entry(entry: Dict) -> List[str]:
    """Create a box on the fly from one schedule entry."""
    t = entry["type"]
    title = entry["stage"]
    p = entry.get("params", {})
    dur = float(entry.get("duration_s", 0))

    lines = []
    if t == "dose":
        lines = [f"Portion: {p.get('portion_L', 0.0):.3f} L"]
    elif t == "mix":
        lines = [f"RPM: {p.get('rpm', 0)}", f"Duration: {fmt_time(dur)}"]
    elif t == "usage":
        lines = [f"Duration: {fmt_time(dur)}"]
    elif t == "collecting":
        lines = [f"Volume: {p.get('volume_L', 0.0):.3f} L"]
    elif t == "settling":
        lines = [f"Duration: {fmt_time(dur)}"]
    elif t == "sep":
        lines = [f"Volume: {p.get('volume_L', 0.0):.3f} L"]
    
    return make_box(title, lines)

def get_flowchart_text(schedule: List[Dict]) -> str:
    """
    Render a vertical ASCII flowchart and RETURN it as a string.
    This avoids output stream issues in multiprocess environments.
    """
    out: List[str] = []
    for i, e in enumerate(schedule):
        out.extend(_render_box_from_entry(e))
        if i < len(schedule) - 1:
            # Add a centered arrow
            out.append(" " * 4 + "↓")
    return "\n".join(out)

# === Usage ===

print("\n=== ASCII FLOWCHART ===\n")
# Capture the text first
flowchart_output = get_flowchart_text(schedule)
# Print it in one go to ensure the block stays together in the logs
print(flowchart_output, flush=True)

# Convert schedule to GeneralRecipe XML

xml_path = save_general_recipe_xml(schedule, order)

---

## Transforming a General Recipe into Semantic Reaction Rules


This pipeline parses a **GeneralRecipe (XML)** into **JSON** and derives a normalized rule set (`reaction_rules_df`) that captures executable material transformations. Each row describes one **processable reaction step**, parameterized by the operation.



### Rule Schema (per row)


* **Inputs** — Required input material(s), comma-separated in defined order.

* **Reaction Type** — Operation that effects the change (e.g., `Mix`, `Usage`).

* **Reaction Param** — Operation parameter (numeric or symbolic; units as specified in the recipe), used for matching/dispatch.

* **Result** — Output material produced by this step (unique label).

In [ ]:
from supporting_scripts.ModPlant_General_Recipe_To_Json import save_parsed_recipe_json_by_id
from supporting_scripts.ModPlant_Reaction_Rules import generate_reaction_rules_from_general_recipe_json, rules_to_dataframe

# # Use input() to ask for file path manually
# xml_path = input("Please enter the path to the General Recipe XML file: ").strip()
# if not os.path.exists(xml_path):
#     raise FileNotFoundError(f"File not found: {xml_path}")

# Parse the GeneralRecipe XML to JSON
json_path = save_parsed_recipe_json_by_id(xml_path)

# Generate reaction rules from the parsed JSON
reaction_rules = generate_reaction_rules_from_general_recipe_json(json_path)
reaction_rules_df = rules_to_dataframe(reaction_rules)

tools.display_dataframe_to_user(name="Semantic Reaction Rules", dataframe=reaction_rules_df)

---

## Typedefs for State and Transition Representation

This section introduces type aliases to make the code more **readable** and **self-documenting** when dealing with ModPlant states and transitions.

* **`ModPlantName = str`**
  Identifier of a ModPlant (e.g., `"HC10"`).

* **`MaterialStr = str`**
  String representation of a material (e.g., `"A : 10.0 litre"`).

* **`ContentTuple = Tuple[str, Tuple[MaterialStr, ...]]`**
  A labeled container for contents, typically:
  `("content", (<material_1>, <material_2>, ...))`.

* **`OutputsTuple = Tuple[str, Tuple[Tuple[str, str], ...]]`**
  Encodes all output ports of a ModPlant and their connections:
  `("outputs", ((port_name, connected_to), ...))`.

* **`ModPlantStateTuple = Tuple[ModPlantName, ContentTuple, OutputsTuple]`**
  The complete state of a single ModPlant:
  `(modplant_name, content_info, outputs_info)`.

* **`FullState = Tuple[ModPlantStateTuple, ...]`**
  The global system state, containing all ModPlant states in an immutable tuple.

* **`Transition = Tuple[int, FullState, str, int, FullState, float, float]`**
  A directed edge in the state graph:
  `(from_state_id, from_state, operation_label, to_state_id, to_state, cost, duration)`.

In [ ]:
from typing import List, Tuple, Set, Dict, Union, Optional
from collections import deque
import concurrent.futures
import pandas as pd
# ========== Typedefs ==========

ModPlantName = str
MaterialStr = str
ContentTuple = Tuple[str, Tuple[MaterialStr, ...]]
# outputs: (key, ((port_name, connected_to, material), ...))
OutputsTuple = Tuple[str, Tuple[Tuple[str, str, str], ...]]
ModPlantStateTuple = Tuple[ModPlantName, ContentTuple, OutputsTuple]
FullState = Tuple[ModPlantStateTuple, ...]
# Transition in the final graph (with IDs)
Transition = Tuple[int, FullState, str, int, FullState, float, float]

# Pure-function stage candidate transition: no IDs, only from_state to to_state
CandidateTransition = Tuple[
    FullState,  # from_state
    FullState,  # to_state
    str,        # operation string
    float,      # cost
    float,      # duration
]

SymbolicVars = Tuple[Tuple[str, float, float], ...]

# Global objects expected to exist outside this module:
# modplant_ops, reaction_rules_df, modplant_maximum_volume, end_state

## Symbolic Variable Utilities
Helpers for reading symbolic bounds from __VARS__ and parsing symbolic material strings.


In [ ]:
# Helper to get vars from state
def get_symbolic_vars(state: FullState) -> List[Tuple[str, float, float]]:
    if state and state[-1][0] == "__VARS__":
        # stored in content: ("vars", ("x:0:10", ...))
        content = state[-1][1][1]
        vars_list = []
        for s in content:
            # format "x:min:max"
            parts = s.split(":")
            vars_list.append((parts[0], float(parts[1]), float(parts[2])))
        return vars_list
    return []

def remove_symbolic_vars(state: FullState) -> FullState:
    if state and state[-1][0] == "__VARS__":
        return state[:-1]
    return state

# Helper to parse symbolic material string
# Format: "Material : 10.0 + -1.0 x litre" OR "Material : 10.0 litre"
def parse_material_string_symbolic(material_str: str) -> Dict[str, Any]:
    """
    Returns dict: { 'Material': {'base': 10.0, 'vars': {'x': -1.0}} }
    """
    result = {}
    entries = material_str.split(',')
    for entry in entries:
        if ':' not in entry: continue
        name, qty_part = entry.split(':', 1)
        name = name.strip()
        qty_part = qty_part.strip()
        
        # Check for symbolic marker "x" (simplification)
        # Regex to parse "10.0 + -1.0 x litre" or "10.0 litre"
        # We assume format: "{base} + {coeff} {var} {unit}" or "{base} {unit}"
        
        base_val = 0.0
        vars_dict = {}
        
        # Simple parsing logic
        tokens = qty_part.split() # ['10.0', '+', '-1.0', 'x', 'litre'] or ['10.0', 'litre']
        
        try:
            if len(tokens) >= 5 and tokens[1] == '+' and tokens[3] in ['x', 'y', 'z']:
                base_val = float(tokens[0])
                coeff = float(tokens[2])
                var_name = tokens[3]
                vars_dict[var_name] = coeff
            else:
                base_val = float(tokens[0])
        except:
            continue
            
        result[name] = {'base': base_val, 'vars': vars_dict}
    return result

def get_symbolic_coeffs(state: FullState) -> Tuple[float, float]:
    """Extract the cost and duration coefficients (multipliers of x) from a symbolic state."""
    cost_c = 0.0
    dur_c = 0.0
    if state and state[-1][0] == "__VARS__":
        # state[-1] structure: ('__VARS__', ('vars', ...), ('coeffs', ('cost:6.0', 'dur:1.0')))
        try:
            # Find the tuple named 'coeffs'
            coeffs_tuple = None
            for item in state[-1]:
                if isinstance(item, tuple) and item[0] == "coeffs":
                    coeffs_tuple = item[1]
                    break
            
            if coeffs_tuple:
                for s in coeffs_tuple:
                    if s.startswith("cost:"):
                        cost_c = float(s.split(":")[1])
                    elif s.startswith("dur:"):
                        dur_c = float(s.split(":")[1])
        except:
            pass
    return cost_c, dur_c

## Utility Functions for Cost Lookup and Transition Recording

This snippet provides two helpers for a graph/search engine that explores ModPlant states.

### `get_operation_cost(...) → int`

Looks up the **per-operation cost** for a given ModPlant.

* **Parameters**

  * `modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]]` — mapping from ModPlant name to its available operations.
    Each operation is a tuple: `(operation_name, operation_param, cost)`.
  * `wb_name: str` — the ModPlant identifier (e.g., `"HC10"`).
  * `operation: str` — operation name to look up (e.g., `"Filling"`).
* **Return**

  * `int` cost associated with the operation on that ModPlant; returns **0** if the ModPlant or operation is not found.
* **Notes**

  * Performs a linear scan of the ModPlant’s operation list.
  * Safe default of `0` prevents crashes when an operation is missing.

### `add_transition(...) → None`

Appends a **state transition** to a shared list, optionally ensuring **no duplicates**.

* **Parameters**

  * `from_id: int` — numeric ID of the source state.
  * `from_state: FullState` — the full (immutable) source state object.
  * `op_str: str` — human-readable operation label for this transition (e.g., `"Mix 200 rpm in HC10"`).
  * `to_id: int` — numeric ID of the destination state.
  * `to_state: FullState` — the full destination state object.
  * `transition_list: List[Transition]` — global accumulator of transitions.
    Each transition appended as `(from_id, from_state, op_str, to_id, to_state, cost, duration)`.
  * `existing_transitions: Set[Tuple[int, str, int, FullState]]` — a deduplication set keyed by `(from_id, op_str, to_id, to_state)`.
  * `cost: float` — edge cost (e.g., effort/energy/time weight).
  * `duration: float` — edge duration (e.g., seconds/minutes).
  * `check: bool` — if `True`, only add when the key is not already present; if `False`, always append.

* **Behavior**

  * When `check=True`, the function builds a **dedupe key** and inserts both the transition and the key atomically.
  * When `check=False`, it **unconditionally** appends (useful for debugging or when parallel edges are intentional).

* **Why both `transition_list` and `existing_transitions`?**

  * `transition_list` preserves **insertion order** for downstream traversal/export.
  * `existing_transitions` provides **O(1)** membership tests to prevent duplicates during BFS/DFS expansion.


In [ ]:
# ========== General Utility Functions ==========
import math
from typing import Dict

def get_operation_cost(
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    wb_name: str,
    operation: str
) -> int:
    """Extract cost (3rd value) of a given operation for a specific ModPlant."""
    for op_name, _, cost in modplant_ops.get(wb_name, []):
        if op_name == operation:
            return cost
    return 0


def parse_material_string(material_str: str) -> Dict[str, float]:
    """Parse 'A: 1.0 litre' -> {'A': 1.0}. Returns empty if symbolic."""
    # if "x" in material_str: return {} # Skip symbolic for standard parsing
    # ... (Original code) ...
    result: Dict[str, float] = {}
    entries = material_str.split(',')
    for entry in entries:
        if ':' not in entry: continue
        name, qty = entry.split(':', 1)
        name = name.strip()
        try:
            amount = float(qty.strip().split()[0])
            result[name] = amount
        except ValueError:
            continue
    return result


def get_material_dict(contents: Tuple[str, ...]) -> Dict[str, float]:
    """Convert tuple of material strings to a material->amount dict."""
    return parse_material_string(', '.join(contents))


def parse_total_volume(content: Tuple[str, Tuple[str, ...]]) -> float:
    """Sum up all numeric quantities in a material content tuple."""
    total = 0.0
    for entry in content[1]:
        try:
            qty_str = entry.split(":")[1].strip().split(" ")[0]
            total += float(qty_str)
        except Exception:
            continue
    return total

def are_materials_equal(d1: Dict[str, float], d2: Dict[str, float], tol=1e-4) -> bool:
    """Performs a fuzzy match between two material dictionaries."""
    if set(d1.keys()) != set(d2.keys()):
        return False
        
    for k, v in d1.items():
        # If the target value is 0 (e.g., volume-less definition from 'Usage'), 
        # ignore numerical comparison; matching the name is sufficient.
        if abs(d2[k]) < 1e-9: 
            continue
            
        if not math.isclose(v, d2[k], abs_tol=tol):
            return False
            
    return True

## Connect Helper Functions and Modular Connection Operations

This section provides helper functions and operations to model **dynamic port connections** between ModPlant. Connections define how material flows from one ModPlant’s output port to another ModPlant’s input port.


### Helper Functions

* **`is_input_free(state, target_wb, target_port) → bool`**
  Checks if a given input port is unoccupied (i.e., no output in the system is connected to it).

* **`get_free_outputs(state, wb_name) → List[str]`**
  Returns all currently unused output ports of the specified ModPlant.

* **`get_all_input_ports(wb_name, modplant_interfaces) → List[str]`**
  Retrieves all defined input port names for a ModPlant from the static interface definition.

* **`get_connected_port_pair(state, from_wb, to_wb) → Tuple[str, str]`**
  Finds an existing connection between one of `from_wb`’s outputs and an input of `to_wb`. Returns `(out_port, in_port)` if found, otherwise `("", "")`.

* **`get_free_output_port(state, wb) → str`**
  Returns one free output port of a given ModPlant (or `""` if none).

* **`get_free_input_port(wb_name, state, modplant_interfaces) → str`**
  Returns one free input port of a ModPlant by excluding those already connected in the state.

* **`update_connection(state, wb_from, out_port, wb_to, in_port) → FullState`**
  Creates a new state where the selected output port of `wb_from` is connected to the specified input of `wb_to`.

  * Ports are updated immutably.
  * Returns a new `FullState` with the modified connection.


### Modular Connection Operations

* **`set_connection(...) → Union[Tuple[FullState, int], None]`**
  Attempts to connect an output port to an input port.

  * Fails immediately if the input port is already in use.
  * Builds the new state via `update_connection`.
  * Adds the state to the **visited registry**, state list, and queue if unseen.
  * Computes **connection cost** by summing `Connect` costs of both source and target ModPlant.
  * Appends a transition with a fixed duration (3).
  * Returns `(new_state, new_id)` if successful.

* **`check_connection(...) → None`**
  Expands all possible **connect transitions** for a given ModPlant:

  * Iterates over all free outputs of the ModPlant.
  * For each free output, considers every other ModPlant and its input ports.
  * Calls `set_connection` to create transitions wherever valid.

In [ ]:
# ========== Connect Helper Functions ==========

def is_input_free(state: FullState, target_wb: str, target_port: str) -> bool:
    """Check if a given input port on target_wb is free (not referenced by any output)."""
    for _, _, (_, outputs) in state:
        for _, connected_to, _ in outputs:
            if connected_to == target_port:
                return False
    return True


def get_free_outputs(state: FullState, wb_name: str) -> List[str]:
    """Return list of free (unconnected) output ports of a ModPlant."""
    for wb, _, (_, outputs) in state:
        if wb == wb_name:
            return [port_name for port_name, connected, material in outputs if not connected]
    return []


def get_all_input_ports(
    wb_name: str,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]]
) -> List[str]:
    """Return all input port names for a given ModPlant based on interface definitions."""
    return [
        port_name
        for port_type, port_name in modplant_interfaces.get(wb_name, [])
        if port_type == "Input"
    ]


def get_connected_port_pair(
    state: FullState,
    from_wb: str,
    to_wb: str,
    material: str = ""
) -> Tuple[str, str]:
    """
    Find an existing connection from 'from_wb' to 'to_wb'.
    Returns (out_port, in_port), optionally filtered by material.
    """
    for wb, _, (_, outputs) in state:
        if wb == from_wb:
            for port_name, conn, mat in outputs:
                if conn.startswith(to_wb + ":") and (not material or mat == material):
                    return port_name, conn.split(":")[1]
    return "", ""


def get_free_output_port(state: FullState, wb: str) -> str:
    """Return a single free output port of 'wb', or empty string if none is free."""
    for wb_name, _, (_, outputs) in state:
        if wb_name == wb:
            for port_name, conn, material in outputs:
                if not conn:
                    return port_name
    return ""


def get_free_input_port(
    wb_name: str,
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]]
) -> str:
    """Return a single free input port of 'wb_name', or empty string if none is free."""
    used_ports = set()
    for _, _, (_, outputs) in state:
        for _, conn, material in outputs:
            if conn.startswith(wb_name + ":"):
                used_ports.add(conn.split(":")[1])
    return next(
        (
            port
            for t, port in modplant_interfaces[wb_name]
            if t == "Input" and port not in used_ports
        ),
        ""
    )


def update_connection(
    state: FullState,
    wb_from: str,
    out_port: str,
    wb_to: str,
    in_port: str,
    material: str
) -> FullState:
    """Return a new state where wb_from:out_port is connected to wb_to:in_port for material."""
    new_state: List[ModPlantStateTuple] = []
    target_str = wb_to + ":" + in_port
    for wb, (content_key, content_val), (out_key, outputs) in state:
        if wb != wb_from:
            new_state.append((wb, (content_key, content_val), (out_key, outputs)))
        else:
            new_outputs = tuple(
                (name, target_str, material) if name == out_port else (name, conn, mat)
                for name, conn, mat in outputs
            )
            new_state.append((wb, (content_key, content_val), (out_key, new_outputs)))
    return tuple(sorted(new_state))

# ========== Pure Connect Operation ==========

def pure_set_connection(
    state: FullState,
    wb_from: str,
    out_port: str,
    wb_to: str,
    in_port: str,
    material: str,
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
) -> Optional[Tuple[FullState, CandidateTransition]]:
    """
    Pure version of 'set_connection':
    - does NOT touch visited / queue / transition_list
    - returns (new_state, candidate_transition) or None if no connection possible
    """
    if not is_input_free(state, wb_to, in_port):
        return None

    new_state = update_connection(state, wb_from, out_port, wb_to, in_port, material)
    op_str = f"Connect({out_port} -> {in_port}) for {material}"

    cost1 = get_operation_cost(modplant_ops, wb_from, "Connect")
    cost2 = get_operation_cost(modplant_ops, wb_to, "Connect")
    cost = float(cost1 + cost2)
    duration = 3.0

    cand: CandidateTransition = (state, new_state, op_str, cost, duration)
    return new_state, cand


# ========== Content Update Helpers (already pure) ==========

def apply_draining(state: FullState, wb: str) -> FullState:
    """Return a new state where all material is removed from ModPlant 'wb'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w == wb:
            new_state.append((w, (content_key, ()), (out_key, outputs)))
        else:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
    return tuple(sorted(new_state))


def apply_filling(state: FullState, wb: str, new_content: Tuple[str, ...]) -> FullState:
    """Return a new state where ModPlant 'wb' content is replaced by 'new_content'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w == wb:
            new_state.append((w, (content_key, new_content), (out_key, outputs)))
        else:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
    return tuple(sorted(new_state))


def apply_partial_draining(
    state: FullState,
    wb: str,
    to_drain: Dict[str, float]
) -> FullState:
    """Return a new state where specified material quantities are removed from ModPlant 'wb'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w != wb:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
            continue

        current_dict = get_material_dict(content_val)
        for k, v in to_drain.items():
            if k in current_dict:
                current_dict[k] -= v
                if current_dict[k] <= 0:
                    del current_dict[k]
        new_content = tuple(f"{k}: {v} litre" for k, v in current_dict.items())
        new_state.append((w, (content_key, new_content), (out_key, outputs)))
    return tuple(sorted(new_state))


def apply_partial_filling(
    state: FullState,
    wb: str,
    to_fill: Dict[str, float]
) -> FullState:
    """Return a new state where specified material quantities are added to ModPlant 'wb'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w != wb:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
            continue

        current_dict = get_material_dict(content_val)
        for k, v in to_fill.items():
            current_dict[k] = current_dict.get(k, 0) + v
        new_content = tuple(f"{k}: {v} litre" for k, v in current_dict.items())
        new_state.append((w, (content_key, new_content), (out_key, outputs)))
    return tuple(sorted(new_state))

## Transfer Operation and Helper Functions

This section introduces functions for handling **material transfer** between ModPlant, including draining one ModPlant, filling another, and updating the state graph with costs and durations.


### Helper Functions

* **`parse_total_volume(content) → float`**
  Parses a ModPlant’s content tuple and sums all material quantities to compute the total volume.

  * Input: `("content", ("A : 10.0 litre", "B : 5.0 litre"))`
  * Output: `15.0`

* **`apply_draining(state, wb) → FullState`**
  Returns a new state where the specified ModPlant has been drained (its contents emptied).

* **`apply_filling(state, wb, new_content) → FullState`**
  Returns a new state where the specified ModPlant is filled with the given content tuple.


### `check_transfer(...) → None`

Generates **transfer transitions** between pairs of ModPlant, following process constraints:

1. **Eligibility Checks**

   * Source ModPlant (`wb1`) must contain material (`vol1 > 0`) and support `Draining`.
   * Target ModPlant (`wb2`) must be empty (`vol2 == 0`), have enough capacity (`max_capacity ≥ vol1`), and support `Filling`.

2. **Connection Handling**

   * If no connection exists between `wb1` and `wb2`, attempt to establish one using `set_connection`.
   * Requires one free output from `wb1` and one free input on `wb2`.

3. **State Update**

   * Apply draining to the source ModPlant (`apply_draining`).
   * Apply filling to the target ModPlant (`apply_filling`) with the original source contents.

4. **Cost and Duration Calculation**

   * **Speed**: Minimum of draining speed (from `wb1`) and filling speed (from `wb2`).
   * **Duration**: `vol1 / transfer_speed` (rounded to 2 decimals).
   * **Cost**: Sum of draining cost (source) and filling cost (target).

5. **Transition Creation**

   * Builds an operation string (e.g.,
     `"Transfering (HC10 → HC20), A : 10.0 litre : Open Valve of HC10_Out1 only, Draining(HC10), Filling(HC20), A : 10.0 litre"`).
   * Adds the new filled state to the **visited registry**, state list, and queue if unseen.
   * Appends a transition `(from_id, state, operation_str, new_id, new_state, cost, duration)` to the transition list.

In [ ]:

# ========== Pure Transfer Operation Candidates ==========

def check_transfer_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
) -> List[CandidateTransition]:
    """
    Pure version of 'check_transfer':
    - does not modify visited / state_list / queue / transition_list
    - returns a list of (from_state, to_state, op_str, cost, duration)
    - semantically matches "Connect + Draining + Filling"
    """
    results: List[CandidateTransition] = []

    for wb1, (content_key1, contents1), (_, _) in state:
        vol1 = parse_total_volume((content_key1, contents1))
        if vol1 <= 0:
            continue
        if not any(op[0] == 'Draining' for op in modplant_ops.get(wb1, [])):
            continue

        # Material name for connection (first material entry)
        material_name = ""
        if contents1:
            first_material = contents1[0].split(':')[0].strip()
            material_name = first_material

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2:
                continue
            vol2 = parse_total_volume((content_key2, contents2))
            if vol2 > 0:
                continue
            max_capacity = modplant_maximum_volume.get(wb2, [0])[0]
            if max_capacity < vol1:
                continue
            if not any(op[0] == 'Filling' for op in modplant_ops.get(wb2, [])):
                continue

            # Check if connection already exists
            out_port, in_port = get_connected_port_pair(state, wb1, wb2, material_name)

            local_state = state
            local_cands: List[CandidateTransition] = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port:
                    continue

                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, material_name, modplant_ops
                )
                if conn_result is None:
                    continue
                local_state, connect_cand = conn_result
                # First add the Connect candidate
                local_cands.append(connect_cand)

            # On local_state: perform Transfer (Drain + Fill)
            state_drained = apply_draining(local_state, wb1)
            state_filled = apply_filling(state_drained, wb2, contents1)

            drain_speed = next(
                (float(op[1]) for op in modplant_ops[wb1] if op[0] == "Draining"),
                0.01
            )
            fill_speed = next(
                (float(op[1]) for op in modplant_ops[wb2] if op[0] == "Filling"),
                0.01
            )
            transfer_speed = min(drain_speed, fill_speed)
            duration = round(vol1 / transfer_speed, 2)

            drain_cost = next((op[2] for op in modplant_ops[wb1] if op[0] == "Draining"), 0) * vol1
            fill_cost = next((op[2] for op in modplant_ops[wb2] if op[0] == "Filling"), 0) * vol1
            cost = float(drain_cost + fill_cost)

            transfer_op = (
                f"Dosing: Open Valve of {out_port} only, "
                f"Draining({wb1}), Filling({wb2}), "
                f"({material_name}: {vol1:.1f} litre)"
            )


            local_cands.append((local_state, state_filled, transfer_op, cost, duration))
            results.extend(local_cands)

    return results


## Partial Transfer Candidate Generator
Creates symbolic partial-transfer candidates with optional connection setup, costing, and duration coefficients.


In [ ]:
def check_transfer_part_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
) -> List[CandidateTransition]:
    results: List[CandidateTransition] = []
    
    if get_symbolic_vars(state):
        return []

    for wb1, (content_key1, contents1), (_, _) in state:
        if wb1 == "__VARS__": continue
        mat_dict1 = get_material_dict(contents1)
        if not mat_dict1: continue
        
        # Assume transferring the first material
        material_name = next(iter(mat_dict1.keys()))
        available_amount = mat_dict1.get(material_name, 0.0)
        
        if available_amount <= 1e-6: continue
        if not any(op[0] == "Draining" for op in modplant_ops.get(wb1, [])): continue

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2 or wb2 == "__VARS__": continue
            
            mat_dict2 = get_material_dict(contents2)
            if mat_dict2 and (len(mat_dict2) != 1 or material_name not in mat_dict2):
                continue

            max_capacity = modplant_maximum_volume.get(wb2, [0.0])[0]
            current_vol2 = sum(mat_dict2.values())
            space_available = max_capacity - current_vol2
            
            if space_available <= 1e-6: continue
            if not any(op[0] == "Filling" for op in modplant_ops.get(wb2, [])): continue

            # Connection logic
            out_port, in_port = get_connected_port_pair(state, wb1, wb2, material_name)
            local_state = state
            local_cands = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port: continue

                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, material_name, modplant_ops
                )
                if conn_result is None: continue
                local_state, connect_cand = conn_result
                local_cands.append(connect_cand)

            # Cost/Duration calculation
            drain_cost_unit = next((float(op[2]) for op in modplant_ops.get(wb1, []) if op[0] == "Draining"), 0.0)
            fill_cost_unit = next((float(op[2]) for op in modplant_ops.get(wb2, []) if op[0] == "Filling"), 0.0)
            cost_coeff = drain_cost_unit + fill_cost_unit
            
            drain_speed = next((float(op[1]) for op in modplant_ops.get(wb1, []) if op[0] == "Draining"), 0.01)
            fill_speed = next((float(op[1]) for op in modplant_ops.get(wb2, []) if op[0] == "Filling"), 0.01)
            transfer_speed = min(drain_speed, fill_speed)
            
            dur_coeff = 1.0 / transfer_speed if transfer_speed > 1e-9 else 0.0

            # Generate state
            max_transfer = min(available_amount, space_available)
            var_name = "x"
            
            new_state_list = []
            for w, (c_key, c_val), (o_key, outputs) in local_state:
                if w == wb1:
                    new_c_list = []
                    for k, v in mat_dict1.items():
                        if k == material_name:
                            new_c_list.append(f"{k}: {v} + -1.0 {var_name} litre")
                        else:
                            new_c_list.append(f"{k}: {v} litre")
                    new_state_list.append((w, (c_key, tuple(sorted(new_c_list))), (o_key, outputs)))
                elif w == wb2:
                    base_v = mat_dict2.get(material_name, 0.0)
                    new_c = (f"{material_name}: {base_v} + 1.0 {var_name} litre",)
                    new_state_list.append((w, (c_key, new_c), (o_key, outputs)))
                else:
                    new_state_list.append((w, (c_key, c_val), (o_key, outputs)))
            
            constraint_entry = (
                "__VARS__", 
                ("vars", (f"{var_name}:0.0:{max_transfer}",)), 
                ("coeffs", (f"cost:{cost_coeff}", f"dur:{dur_coeff}")) 
            )
            new_state_list.append(constraint_entry)
            symbolic_state = tuple(sorted(new_state_list, key=lambda x: x[0]))

            # === Change: append 'for <material>' in the op string ===
            op_str = f"Variable Transfer setup ({wb1}->{wb2}) for {material_name}: {var_name} in [0, {max_transfer:.2f}]"
            
            local_cands.append((local_state, symbolic_state, op_str, 0.0, 0.0))
            
            results.extend(local_cands)

    return results

## Dosing Operation and Material Transfer Helpers

This section implements functions for **partial draining and filling** of ModPlant, supporting precise dosing operations defined by recipe rules.


### Material Parsing

* **`parse_material_string(material_str: str) → Dict[str, float]`**
  Converts a material string (e.g., `"A: 1.0 litre, B: 2.0 litre"`) into a dictionary `{ "A": 1.0, "B": 2.0 }`.

* **`get_material_dict(contents: Tuple[str, ...]) → Dict[str, float]`**
  Converts a tuple of material strings (from ModPlant content state) into a material dictionary using `parse_material_string`.


### State Updates

* **`apply_partial_draining(state, wb, to_drain) → FullState`**
  Removes specified material amounts from a ModPlant.

  * If a material quantity reaches zero, it is removed from the content list.

* **`apply_partial_filling(state, wb, to_fill) → FullState`**
  Adds specified material amounts to a ModPlant.

  * If a material already exists, its quantity is increased; otherwise, it is added.

Both functions return a **new immutable state** with updated contents.


### `dosing_operation(...) → None`

Generates valid **dosing transitions** based on reaction rules:

1. **Source ModPlant (wb1) checks**

   * Must support **Draining**.
   * Must contain at least the required material amounts (`req_dose`).

2. **Target ModPlant (wb2) checks**

   * Must support **Filling**.
   * If the recipe specifies `Inputs`, the ModPlant’s current contents must exactly match them.
   * If no `Inputs` are specified, the ModPlant must be empty.

3. **Connection handling**

   * If no connection exists between `wb1` and `wb2`, attempt to establish one via `set_connection`.

4. **State updates**

   * Apply partial draining on `wb1` (`apply_partial_draining`).
   * Apply partial filling on `wb2` (`apply_partial_filling`).

5. **Cost and duration**

   * **Duration**: calculated as `dose_amount / min(draining_speed, filling_speed)`.
   * **Cost**: sum of draining and filling costs for both ModPlant.

6. **Transition creation**

   * Builds a descriptive operation string (e.g.,
     `"Dosing (HC10 → HC20), (A: 1.0 litre): Open Valve of HC10_Out1 only, Draining(HC10), Filling(HC20), (A: 1.0 litre)"`).
   * Registers the new state if unseen and appends a transition to the global list with cost and duration metadata.

In [ ]:
def compute_dosing_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Dosing candidates for a single state and rule."""
    results: List[CandidateTransition] = []

    req_dose = parse_material_string(rule_row["Reaction Param"])
    req_inputs = parse_material_string(rule_row["Inputs"]) if rule_row["Inputs"] else None

    for wb1, (content_key1, contents1), (_, _) in state:
        if not any(op[0] == "Draining" for op in modplant_ops.get(wb1, [])):
            continue
        wb1_dict = get_material_dict(contents1)
        if any(wb1_dict.get(k, 0) < v for k, v in req_dose.items()):
            continue

        material_name = ", ".join(req_dose.keys()) if req_dose else ""

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2:
                continue
            if not any(op[0] == "Filling" for op in modplant_ops.get(wb2, [])):
                continue
            wb2_dict = get_material_dict(contents2)

            if req_inputs:
                if wb2_dict != req_inputs:
                    continue
            else:
                if wb2_dict:
                    continue

            out_port, in_port = get_connected_port_pair(state, wb1, wb2, material_name)
            local_state = state
            local_cands: List[CandidateTransition] = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port:
                    continue
                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, material_name, modplant_ops
                )
                if conn_result is None:
                    continue
                local_state, connect_cand = conn_result
                local_cands.append(connect_cand)

            drained = apply_partial_draining(local_state, wb1, req_dose)
            filled = apply_partial_filling(drained, wb2, req_dose)

            dose_amount = sum(req_dose.values())
            draining_speed = float(next(
                (op[1] for op in modplant_ops[wb1] if op[0] == "Draining"),
                1.0
            ))
            filling_speed = float(next(
                (op[1] for op in modplant_ops[wb2] if op[0] == "Filling"),
                1.0
            ))
            speed = min(draining_speed, filling_speed) if draining_speed and filling_speed else 1.0
            duration = dose_amount / speed if speed > 0 else 0.0

            cost_draining = next((op[2] for op in modplant_ops[wb1] if op[0] == "Draining"), 0) * dose_amount
            cost_filling = next((op[2] for op in modplant_ops[wb2] if op[0] == "Filling"), 0) * dose_amount
            cost = float(cost_draining + cost_filling)

            # === Change: format volumes to two decimal places ===
            raw_param = rule_row['Reaction Param']
            formatted_parts = []
            if raw_param:
                for p in raw_param.split(','):
                    if ':' in p:
                        name, rest = p.split(':', 1)
                        tokens = rest.strip().split()
                        if tokens:
                            try:
                                # Attempt to parse and keep two decimals
                                val = float(tokens[0])
                                unit = " ".join(tokens[1:])
                                formatted_parts.append(f"{name.strip()}: {val:.2f} {unit}")
                            except ValueError:
                                formatted_parts.append(p.strip())
                        else:
                            formatted_parts.append(p.strip())
                    else:
                        formatted_parts.append(p.strip())
                formatted_param = ", ".join(formatted_parts)
            else:
                formatted_param = ""
            # ==================================

            op_str = (
                f"Dosing: Open Valve of {out_port} only, "
                f"Draining({wb1}), Filling({wb2}), ({formatted_param})"
            )

            local_cands.append((local_state, filled, op_str, cost, duration))
            results.extend(local_cands)

    return results

## Mixing Operation

This function defines how a **mixing step** is executed inside the ModPlant system, transforming input materials into a single result material under specified conditions (RPM and time).


### Workflow

1. **Parse recipe requirements**

   * **Inputs**: Materials required for mixing (`rule_row["Inputs"]`).
   * **Reaction Parameters**: Expected format `"X rpm / Y second"`.

     * Extracts `rpm` (e.g., `"200"`) and `duration` in seconds (e.g., `30`).
   * If parsing fails, the operation is skipped.

2. **Check eligible ModPlant**

   * Iterates over all ModPlant in the current state.
   * Compares the ModPlant’s content against the required inputs — must match exactly.
   * Verifies that the ModPlant supports **Stirring** at the given `rpm` by checking `modplant_ops`.

3. **Build new state**

   * For the chosen ModPlant:

     * Replace its contents with the **result material** (`rule_row["Result"]`).
     * Preserve its output connections.
   * For all other ModPlant:

     * Keep their state unchanged.
   * Construct a new immutable `FullState`.

4. **Operation metadata**

   * **Operation string**: `"Stirring (HC10), 200rpm for 30s"`.
   * **Cost**: Taken from the matching `Stirring` operation in `modplant_ops`.
   * **Duration**: Taken directly from the parsed parameter (e.g., 30).

5. **State registration**

   * If the new state has not been seen, assign it a new ID, add it to `visited`, `state_list`, and enqueue it for further exploration.
   * Otherwise, reuse its existing ID.

6. **Transition creation**

   * Records the transition `(from_state → to_state)` into `transition_list` with cost and duration metadata, avoiding duplicates via `existing_transitions`.


In [ ]:
def format_result_content(result_mat_str: str, total_vol: float, unit: str = "litre") -> str:
    """
    Format the result material string with total volume.
    Returns a string in the format "Material : volume unit"
    """
    if ":" in result_mat_str:
        # Already includes volume
        return result_mat_str
    else:
        # Only material name, append volume
        return f"{result_mat_str} : {total_vol:.1f} {unit}"

## Mixing Candidate Generator
Matches mixing rules to vessel capabilities and produces stirring transitions.


In [ ]:
def compute_mixing_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    results: List[CandidateTransition] = []
    
    # Parse required inputs
    req_inputs = parse_material_string(rule_row["Inputs"])
    if not req_inputs: return []
    
    # Parse reaction parameters (rpm/duration)
    param_str = str(rule_row["Reaction Param"])
    if "/" in param_str:
        parts = param_str.split('/')
        rpm_str = parts[0].strip().split()[0]
        dur_str = parts[1].strip().split()[0]
    else:
        # If no '/', assume only duration provided
        rpm_str = "0"
        dur_str = param_str.strip().split()[0] if param_str.strip() else "0"

    try:
        duration = float(dur_str)
    except:
        duration = 0.0

    for wb, (content_key, contents), (_, _) in state:
        content_dict = get_material_dict(contents)

        # 1. Check materials
        if not are_materials_equal(content_dict, req_inputs, tol=1e-4): continue

        # 2. Check mixing operation with matching rpm
        # Attention: rpm is stored as string in operation
        matching_mix = next(
            (op for op in modplant_ops.get(wb, []) if op[0] == "Stirring" and str(op[1]) == rpm_str),
            None
        )
        #print(f"Checking ModPlant {wb} for Mixing at {rpm_str} rpm: {'Found' if matching_mix else 'Not Found'}")
        if not matching_mix: continue

        cost = float(matching_mix[2])
        
        # 3. Construct new content based on Result
        total_vol = sum(content_dict.values())
        unit = "litre"
        if contents:
            parts = contents[0].split()
            if parts: unit = parts[-1]
            
        result_mat = rule_row["Result"]
        # FIX: Use helper to format result content
        new_content_str = format_result_content(result_mat, total_vol, unit)
        
        new_state_list = []
        for w, (c_key, c_val), (out_key, outputs) in state:
            if w == wb:
                new_state_list.append((w, (c_key, (new_content_str,)), (out_key, outputs)))
            else:
                new_state_list.append((w, (c_key, c_val), (out_key, outputs)))
        new_state = tuple(sorted(new_state_list))

        op_str = f"Stirring ({wb}), {rpm_str}rpm for {duration}s"
        results.append((state, new_state, op_str, cost, duration))
        #print( f"Mixing candidate: {op_str}, cost: {cost}, duration: {duration}" )
    return results

## Usage Operation

This function models **usage steps** from a recipe, where materials in a ModPlant are consumed or transformed into a final product without requiring specialized hardware operations.

### Workflow

1. **Parse recipe requirements**

   * **Inputs**: Expected material(s) (e.g., `"A_B_C_mixed : 6.0 litre"`) parsed into a dict `{ "A_B_C_mixed": 6.0 }`.
   * **Duration**: Taken from `"Reaction Param"` (e.g., `"3600 second"` → `3600.0`). If parsing fails, the operation is skipped.

2. **Check eligible ModPlant**

   * Iterates through all ModPlant in the state.
   * Verifies that the ModPlant’s current contents **exactly match** the required inputs.
   * Ensures the ModPlant supports the `"None"` operation, meaning no special hardware capability is needed.

3. **Build new state**

   * For the selected ModPlant:

     * Contents are replaced with the **result material** (`rule_row["Result"]`).
     * Volume is preserved, but only the label changes.
   * Other ModPlant remain unchanged.
   * Produces a new immutable `FullState`.

4. **Operation metadata**

   * **Operation string**: `"Usage (HC10), 3600s: None"`.
   * **Cost**: Fixed at `0`.
   * **Duration**: Reset to `0` in the current code (though initially parsed from the recipe).

5. **Transition creation**

   * Registers the new state if unseen, otherwise reuses its existing ID.
   * Adds the transition `(from_state → to_state)` to the global transition list with cost and duration metadata.


In [ ]:
def compute_usage_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Usage candidates for a single state and rule."""
    
    #print(f"Computing Usage candidates for rule: {rule_row['Inputs']} -> {rule_row['Result']}")

    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])
    #print(f"Required inputs for Usage: {req_inputs}")

    try:
        duration_str = rule_row["Reaction Param"].strip().split()[0]
        duration_val = float(duration_str)
    except Exception:
        return results

    for wb, (content_key, contents), (_, _) in state:
        content_dict = get_material_dict(contents)
        
        if not are_materials_equal(content_dict, req_inputs): continue
        
        has_none_op = any(op[0] == "None" for op in modplant_ops.get(wb, []))
        if not has_none_op:
            continue
        
        
        result_mat = rule_row["Result"]

        # ========== FIX: Keep total volume ==========
        total_vol = sum(content_dict.values())
        
        unit = "litre"
        if contents:
            parts = contents[0].split()
            if parts: unit = parts[-1]
            
        new_content_str = f"{result_mat} : {total_vol:.2f} {unit}"
        new_content = (new_content_str,)
        # ==================================

        new_state_list: List[ModPlantStateTuple] = []
        for w, (c_key, c_val), (out_key, outputs) in state:
            if w == wb:
                new_state_list.append((w, (c_key, new_content), (out_key, outputs)))
            else:
                new_state_list.append((w, (c_key, c_val), (out_key, outputs)))
        new_state = tuple(sorted(new_state_list))

        op_str = f"Usage ({wb}), {duration_val}s: None"
        results.append((state, new_state, op_str, 0.0, 0.0))

    return results

## Settling Operation

This function models **settling steps** in a recipe, where materials are left undisturbed in a ModPlant for a defined period, leading to a new result material.


### Workflow

1. **Parse recipe requirements**

   * **Inputs**: Extracted from `rule_row["Inputs"]`, parsed into a material dictionary.
   * **Duration**: Taken from `rule_row["Reaction Param"]` (e.g., `"300 second"` → `300.0`).
   * If parsing fails, the operation is skipped.

2. **Check eligible ModPlant**

   * Iterates through all ModPlant in the state.
   * Verifies that the ModPlant’s current contents **exactly match** the required inputs.
   * Ensures the ModPlant supports the `"Settling"` operation (capability defined in `modplant_ops`).

3. **Build new state**

   * For the target ModPlant: replace its contents with the **result material** (`rule_row["Result"]`).
   * All other ModPlant remain unchanged.
   * Produces a new immutable `FullState`.

4. **Operation metadata**

   * **Operation string**: `"Settling (HC10), 300s: Settling"`.
   * **Cost**: Taken from the ModPlant’s `Settling` operation cost in `modplant_ops`.
   * **Duration**: Reset to `0` in the current implementation (even though a recipe value is parsed).

5. **Transition creation**

   * Registers the new state if unseen; otherwise reuses its existing ID.
   * Adds the transition `(from_state → to_state)` with the computed cost and (reset) duration.

In [ ]:


def compute_settling_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Settling candidates for a single state and rule."""
    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])

    try:
        duration_str = rule_row["Reaction Param"].strip().split()[0]
        duration_val = float(duration_str)
    except Exception:
        return results

    for wb, (content_key, contents), (_, _) in state:
        content_dict = get_material_dict(contents)
        if not are_materials_equal(content_dict, req_inputs): continue

        settling_op = next((op for op in modplant_ops.get(wb, []) if op[0] == "Settling"), None)
        if not settling_op:
            continue

        cost = float(settling_op[2])
        result_mat = rule_row["Result"]

        new_state_list: List[ModPlantStateTuple] = []
        for w, (c_key, c_val), (out_key, outputs) in state:
            if w == wb:
                new_content = (f"{result_mat}",)
                new_state_list.append((w, (c_key, new_content), (out_key, outputs)))
            else:
                new_state_list.append((w, (c_key, c_val), (out_key, outputs)))
        new_state = tuple(sorted(new_state_list))

        op_str = f"Settling ({wb}), {duration_val}s: Settling"
        # Original code used duration=0 in the transition, keep this semantics
        results.append((state, new_state, op_str, cost, 0.0))

    return results

## Separation Operation

This function implements recipe-driven **separation steps**, where a component is drained from one ModPlant and transferred to another, leaving the source with the specified result composition.


### Workflow

1. **Parse recipe inputs**

   * **Inputs**: Taken from `rule_row["Inputs"]`, parsed into a dict of required materials.
   * **Result material**: Parsed from `rule_row["Result"]`.
   * **Separated component**: Defined by `rule_row["Reaction Param"]` (e.g., `"C"`).

2. **Check eligible source ModPlant (wb1)**

   * Contents must **exactly match** the required inputs.
   * Must support **Draining**.

3. **Check eligible target ModPlant (wb2)**

   * Cannot be the same as `wb1`.
   * Must be either empty or already contain only the separated component.
   * Must support **Filling**.
   * Must have enough remaining capacity to accept the separated volume.

4. **Volume calculation**

   * `source_volume`: Total volume of required inputs.
   * `target_volume`: Volume of the remaining material (unless `"End"`).
   * `transfer_volume`: Amount of separated component to move (`source_volume - target_volume`).
   * Ensures `transfer_volume` does not exceed remaining target capacity.

5. **Connection handling**

   * If no direct connection exists between `wb1` and `wb2`, attempts to establish one using `set_connection`.

6. **State updates**

   * Apply **partial draining** of the separated component from `wb1`.
   * Apply **partial filling** of the separated component into `wb2`.
   * Update `wb1`’s contents with the **result material** from the recipe.

7. **Cost and duration**

   * **Speed**: Minimum of source draining and target filling speeds.
   * **Duration**: `transfer_volume / speed` (though later reset to `0`).
   * **Cost**: Sum of source draining and target filling costs.

8. **Transition creation**

   * Builds a descriptive operation string (e.g.,
     `"Separation (HC10 → HC20): C: Open Valve of HC10_Out1 only, Draining(HC10), Filling(HC20), (C: 5.00 litre)"`).
   * If `rule_row["Result"] == "End"`, adds a direct transition to the **end\_state**.
   * Otherwise, registers the new state in `visited`, enqueues it, and logs the transition with cost and duration.


In [ ]:
def compute_separation_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Separation candidates for a single state and rule."""
    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])
    result_material = parse_material_string(rule_row["Result"])
    separated_comp = rule_row["Reaction Param"].strip()

    for wb1, (content_key1, contents1), (_, _) in state:
        content_dict1 = get_material_dict(contents1)
        if content_dict1 != req_inputs:
            continue
        if not any(op[0] == "Draining" for op in modplant_ops.get(wb1, [])):
            continue

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2:
                continue
            content_dict2 = get_material_dict(contents2)
            if content_dict2 and (list(content_dict2.keys()) != [separated_comp]):
                continue

            capacity_left = float('inf')
            for op in modplant_ops.get(wb2, []):
                if op[0] == "Filling":
                    capacity_left = modplant_maximum_volume.get(wb2, [0])[0]
                    break
            used_volume = sum(content_dict2.values())
            remaining_capacity = capacity_left - used_volume
            if remaining_capacity <= 0:
                continue

            source_volume = sum(req_inputs.values())
            target_volume = (
                0.0 if rule_row["Result"] == "End" else sum(result_material.values())
            )
            transfer_volume = max(0.0, source_volume - target_volume)
            if transfer_volume > remaining_capacity:
                continue

            out_port, in_port = get_connected_port_pair(state, wb1, wb2, separated_comp)
            local_state = state
            local_cands: List[CandidateTransition] = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port:
                    continue
                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, separated_comp, modplant_ops
                )
                if conn_result is None:
                    continue
                local_state, connect_cand = conn_result
                local_cands.append(connect_cand)

            drain_dict = {separated_comp: transfer_volume}
            drained = apply_partial_draining(local_state, wb1, drain_dict)
            filled = apply_partial_filling(drained, wb2, drain_dict)

            draining_speed = float(next(
                (op[1] for op in modplant_ops[wb1] if op[0] == "Draining"),
                1.0
            ))
            filling_speed = float(next(
                (op[1] for op in modplant_ops[wb2] if op[0] == "Filling"),
                1.0
            ))
            speed = min(draining_speed, filling_speed)
            duration_val = transfer_volume / speed if speed > 0 else 0.0

            cost_draining = next((op[2] for op in modplant_ops[wb1] if op[0] == "Draining"), 0)
            cost_filling = next((op[2] for op in modplant_ops[wb2] if op[0] == "Filling"), 0)
            cost = float(cost_draining + cost_filling)

            final_state_list: List[ModPlantStateTuple] = []
            for w, (c_key, c_val), (out_key, outputs) in filled:
                if w == wb1:
                    new_content = tuple(f"{k}: {v} litre" for k, v in result_material.items())
                    final_state_list.append((w, (c_key, new_content), (out_key, outputs)))
                else:
                    final_state_list.append((w, (c_key, c_val), (out_key, outputs)))
            final_state = tuple(sorted(final_state_list))

            op_str = (
                f"Separation: Open Valve of {out_port} only, "
                f"Draining({wb1}), Filling({wb2}), "
                f"({separated_comp}: {transfer_volume:.2f} litre)"
            )

            if rule_row["Result"] == "End":
                # Direct transition to end_state
                local_cands.append((local_state, end_state, "End, " + op_str, cost, 0.0))
            else:
                local_cands.append((local_state, final_state, op_str, cost, 0.0))

            results.extend(local_cands)

    return results

## Rule Checking and Dispatch

The `check_rules` function applies **semantic reaction rules** to the current system state, expanding all possible process transitions defined in the recipe.


### Workflow

1. **Dispatch mapping**

   * Defines a dictionary that maps **reaction types** to their corresponding handler functions:

     * `"Dosing"` → `dosing_operation`
     * `"Mix"` → `mixing_operation`
     * `"Usage"` → `usage_operation`
     * `"Settling"` → `settling_operation`
     * `"Separation"` → `separation_operation`

2. **Iterating rules**

   * Loops over each row in the `reaction_rules_df` DataFrame (one per recipe step).
   * Extracts the handler function based on the `"Reaction Type"`.

3. **Handler execution**

   * If a handler exists, it is called with the full **search context**:

     * Current state (`state`, `state_id`)
     * Search data structures (`visited`, `state_list`, `queue`, `depth`)
     * Transition records (`transition_list`, `existing_transitions`)
     * Plant model (`modplant_interfaces`, `modplant_ops`)
     * Rule definition (`rule_row`)

4. **State-space expansion**

   * Each handler may create new states, add them to the search frontier, and append transitions with costs/durations.


In [ ]:
def solve_and_instantiate(
    state: FullState,
    wb_name: str,
    required_material: Dict[str, float]
) -> Optional[Tuple[FullState, float]]:
    """
    Resolve symbolic variable x for a specific ModPlant and instantiate a concrete state.
    No persistent cache to keep runs stateless across kernels.
    """
    vars_list = get_symbolic_vars(state)
    if not vars_list:
        return None

    var_name, v_min, v_max = vars_list[0]

    # locate target ModPlant content
    target_content = None
    for w, (_, content), _ in state:
        if w == wb_name:
            target_content = content
            break
    if target_content is None:
        return None

    solved_x = None

    # Branch 1: required_material empty -> solve base + coeff * x = 0
    if not required_material:
        for entry in target_content:
            if var_name not in entry:
                continue
            if "_mixed" in entry:
                continue
            try:
                rhs = entry.split(":", 1)[1]
                tokens = rhs.split()
                if var_name in tokens:
                    idx = tokens.index(var_name)
                    if idx > 0 and idx + 1 < len(tokens):
                        coeff = float(tokens[idx - 1])
                        base = float(tokens[0])
                        if abs(coeff) > 1e-9:
                            solved_x = -base / coeff
                            break
            except Exception:
                continue

    # Branch 2: required_material present -> match quantities exactly
    else:
        current_sym = parse_material_string_symbolic(', '.join(target_content))
        for mat, req_qty in required_material.items():
            if mat not in current_sym:
                return None
            info = current_sym[mat]
            base = info['base']
            coeffs = info['vars']

            if var_name not in coeffs:
                if abs(base - req_qty) > 1e-6:
                    return None
            else:
                coeff = coeffs[var_name]
                if abs(coeff) < 1e-9:
                    if abs(base - req_qty) > 1e-6:
                        return None
                else:
                    val = (req_qty - base) / coeff
                    if solved_x is not None and abs(solved_x - val) > 1e-6:
                        return None
                    solved_x = val

    if solved_x is None:
        return None

    if not (v_min - 1e-6 <= solved_x <= v_max + 1e-6):
        return None
    if solved_x < v_min:
        solved_x = v_min
    if solved_x > v_max:
        solved_x = v_max

    # rebuild only modplants containing var_name
    new_state_list = []
    for w, (c_key, c_val), (o_key, outputs) in state:
        if w == "__VARS__":
            continue

        has_var = any(var_name in line for line in c_val)
        if not has_var:
            new_state_list.append((w, (c_key, c_val), (o_key, outputs)))
            continue

        parsed = parse_material_string_symbolic(', '.join(c_val))
        new_mat_strs = []
        for m, info in parsed.items():
            qty = info['base']
            if var_name in info['vars']:
                qty += info['vars'][var_name] * solved_x
            if abs(qty) < 1e-6:
                continue
            new_mat_strs.append(f"{m}: {qty:.2f} litre")

        new_state_list.append((w, (c_key, tuple(sorted(new_mat_strs))), (o_key, outputs)))

    concrete_state = tuple(sorted(new_state_list, key=lambda x: x[0]))
    return concrete_state, solved_x



## Reaction Rule Candidate Dispatcher
Dispatches between symbolic and concrete rule evaluation to enumerate feasible transitions.


In [ ]:
def check_rules_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    reaction_rules_df: pd.DataFrame,
) -> List[CandidateTransition]:
    results: List[CandidateTransition] = []
    is_symbolic = bool(get_symbolic_vars(state))

    for _, rule_row in reaction_rules_df.iterrows():
        raw_inputs = rule_row["Inputs"]
        req_inputs = parse_material_string(raw_inputs)
        rtype = rule_row["Reaction Type"]
        
        # Pre-parse Mix rpm so symbolic mode skips unsupported vessels quickly
        rpm_required = None
        if rtype == "Mix":
            param_str = str(rule_row["Reaction Param"])
            if "/" in param_str:
                rpm_required = param_str.split('/')[0].strip().split()[0]
            else:
                tokens = param_str.strip().split()
                rpm_required = tokens[0] if tokens else None
        
        # === Path A: symbolic state ===
        if is_symbolic:
            # if is_symbolic and rule_row["Reaction Type"] != "Dosing":
            #     continue
            cost_coeff, dur_coeff = get_symbolic_coeffs(state)
            
            for wb, _, _ in state:
                if wb == "__VARS__": continue
                
                if rpm_required is not None:
                    ops = modplant_ops.get(wb, [])
                    if not any(op[0] == "Stirring" and str(op[1]) == rpm_required for op in ops):
                        continue
                
                sol = solve_and_instantiate(state, wb, req_inputs)
                if sol:
                    concrete_state, solved_x = sol
                    
                    resolve_cost = cost_coeff * solved_x
                    resolve_dur = dur_coeff * solved_x
                    
                    # === Change: include material name in Resolve description ===
                    mat_name = list(req_inputs.keys())[0] if req_inputs else "Unknown"
                    op_str = f"Resolve(x={solved_x:.2f}) for {mat_name}"
                    
                    results.append((state, concrete_state, op_str, resolve_cost, resolve_dur))
            continue

        # === Path B: concrete state (original logic) ===
        candidates = []
        
        if rtype == "Dosing":
            candidates = compute_dosing_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Mix":
            candidates = compute_mixing_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Usage":
            candidates = compute_usage_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Settling":
            candidates = compute_settling_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Separation":
            candidates = compute_separation_candidates(state, modplant_interfaces, modplant_ops, rule_row)
            
        for (cand_from_state, to_state, op_str, op_cost, op_dur) in candidates:
            # Detect implicit connection changes
            has_conn_change = False
            conn_op_str = ""
            conn_cost = 0.0
            conn_dur = 0.0
            
            if len(state) == len(cand_from_state):
                 for i in range(len(state)):
                    w1, _, out1 = state[i]
                    w2, _, out2 = cand_from_state[i]
                    if w1 == "__VARS__" or w2 == "__VARS__": continue
                    
                    if out1 != out2:
                        has_conn_change = True
                        for p_idx, port_def in enumerate(out2[1]):
                            old_def = out1[1][p_idx]
                            if port_def != old_def and port_def[1] != "":
                                target_full = port_def[1]
                                target_clean = target_full.split(":")[1] if ":" in target_full else target_full
                                conn_op_str += f"Connect({port_def[0]} -> {target_clean}) for {port_def[2]} & "
                                conn_cost += 2.0 
                                conn_dur += 3.0
            
            if has_conn_change:
                conn_op_str = conn_op_str.rstrip(" & ")
                results.append((state, cand_from_state, conn_op_str, conn_cost, conn_dur))
            else:
                results.append((state, to_state, op_str, op_cost, op_dur))

    return results



---

## BFS Core: State-Space Exploration

This function implements the **Breadth-First Search (BFS)** procedure to explore all feasible process states and transitions for a modular plant system.


### `run_bfs(start_state, modplant_interfaces, max_steps)`

1. **Initialization**

   * `visited`: Maps each unique `FullState` to its assigned ID.
   * `state_list`: List of all discovered states in insertion order.
   * `transition_list`: Records transitions as `(from_id, from_state, operation, to_id, to_state, cost, duration)`.
   * `existing_transitions`: Set used for duplicate prevention.
   * `queue`: BFS frontier, seeded with both the **start state** (ID 0) and the **end state** (ID 1).

2. **Main loop**

   * Pops `(state, id, depth)` from the queue.
   * Stops expanding states if `depth ≥ max_steps`.
   * Skips expansions for the **end state** (ID 1).
   * For each ModPlant in the current state:

     * **Transfer operations** are checked via `check_transfer`.
     * **Reaction rules** (Dosing, Mixing, Usage, Settling, Separation) are applied via `check_rules`.
     * (Optional) **Connection operations** can be enabled via `check_connection`.

3. **Termination**

   * When BFS completes, returns both:

     * `transition_list` (all valid edges in the state graph).
     * `state_list` (all visited states).


### Post-processing

* **State indexing**:

  * `state_index`: Maps state → ID.
  * `index_state`: Maps ID → state.

* **End-state verification**:

  * Confirms at least one transition leads to the `end_state`.
  * If none exist, raises `"No feasible solution found"`.

* **Visualization**:

  * Builds a DataFrame summarizing all transitions, with columns:

    * **From\_ID**, **From\_State**, **Operation**, **Cost**, **Duration**, **To\_ID**, **To\_State**
  * Displays the table for inspection or debugging.

In [ ]:
# ========== Pure Expansion for a Single State ==========
def expand_state_pure(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    reaction_rules_df: pd.DataFrame,
) -> List[CandidateTransition]:
    """
    Pure expansion of a single state:
    - Transfer operations
    - Reaction Rules (Dosing / Mix / Usage / Settling / Separation)
    """
    candidates: List[CandidateTransition] = []
    #candidates.extend(check_transfer_candidates(state, modplant_interfaces, modplant_ops))
    candidates.extend(check_transfer_part_candidates(state, modplant_interfaces, modplant_ops))
    candidates.extend(check_rules_candidates(state, modplant_interfaces, modplant_ops, reaction_rules_df))
    return candidates


# ========== Multi-threaded BFS Core (free-threading friendly) ==========

def run_bfs(
    start_state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    max_steps: Optional[int] = None,
    num_threads: int = 8,
    batch_size: Optional[int] = None,
) -> Tuple[List[Transition], List[FullState]]:
    """
    Multi-threaded BFS:
    - Uses a thread pool to expand states via expand_state_pure (pure function).
    - Only the main thread modifies visited / state_list / queue / transition_list.
    - max_steps: if not None, limit BFS depth; otherwise, run without a depth limit.
    """
    if batch_size is None:
        batch_size = num_threads

    visited: Dict[FullState, int] = {}
    state_list: List[FullState] = []
    transition_list: List[Transition] = []
    existing_transitions: Set[Tuple[int, str, int, FullState]] = set()
    queue: deque = deque()

    # Initialize start / end states
    start_id = 0
    visited[start_state] = start_id
    state_list.append(start_state)
    queue.append((start_state, start_id, 0))

    end_id = 1
    visited[end_state] = end_id
    state_list.append(end_state)
    # end_state is not expanded
    queue.append((end_state, end_id, 0))

    def add_transition(
        from_id: int,
        from_state: FullState,
        op_str: str,
        to_id: int,
        to_state: FullState,
        cost: float,
        duration: float,
    ) -> None:
        key = (from_id, op_str, to_id, to_state)
        if key in existing_transitions:
            return
        transition_list.append((from_id, from_state, op_str, to_id, to_state, cost, duration))
        existing_transitions.add(key)

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        while queue:
            batch: List[Tuple[FullState, int, int]] = []

            # Collect a batch of states from the BFS frontier
            while queue and len(batch) < batch_size:
                current_state, current_id, depth = queue.popleft()
                if current_id == end_id:
                    continue
                if max_steps is not None and depth >= max_steps:
                    continue
                batch.append((current_state, current_id, depth))

            if not batch:
                if not queue:
                    break
                continue

            futures: List[Tuple[concurrent.futures.Future, FullState, int, int]] = []
            for current_state, current_id, depth in batch:
                fut = executor.submit(
                    expand_state_pure,
                    current_state,
                    modplant_interfaces,
                    modplant_ops,
                    reaction_rules_df,
                )
                futures.append((fut, current_state, current_id, depth))

            # Collect results and merge into global BFS structures
            for fut, current_state, current_id, depth in futures:
                try:
                    cands = fut.result()
                except Exception as e:
                    print(f"Error expanding state: {e}")
                    continue

                for from_state, to_state, op_str, cost, duration in cands:
                    # Ensure from_state is in visited
                    if from_state not in visited:
                        new_from_id = len(state_list)
                        visited[from_state] = new_from_id
                        state_list.append(from_state)
                        queue.append((from_state, new_from_id, depth + 1))
                    from_id = visited[from_state]

                    # Ensure to_state is in visited
                    if to_state not in visited:
                        new_to_id = len(state_list)
                        visited[to_state] = new_to_id
                        state_list.append(to_state)
                        queue.append((to_state, new_to_id, depth + 1))
                    to_id = visited[to_state]

                    add_transition(from_id, from_state, op_str, to_id, to_state, cost, duration)

    return transition_list, state_list

# ========== Run multi-threaded BFS and collect graph ==========
current_threads = 0
current_batch_size = 0

# Get CPU core count (default to 1 if detection fails)
import os
cpu_cores = os.cpu_count() or 1

# Calculate dynamic thread count: Total Cores (reserve for OS), minimum 1
dynamic_threads = max(1, cpu_cores)
if dynamic_threads > 10:
    dynamic_threads -= 2 # Reserve 2 cores for OS if many cores are available

if sys.platform == "win32":
    print("Detected OS: Windows")
    current_threads = dynamic_threads
    current_batch_size = current_threads * 64 if dynamic_threads >= 14 else current_threads * 32
elif sys.platform == "darwin":
    print("Detected OS: macOS")
    current_threads = cpu_cores
    current_batch_size = current_threads * 8
else:
    current_threads = dynamic_threads
    current_batch_size = current_threads * 8

print(f"Using {current_threads} threads with batch size {current_batch_size}")

# Pre-check: ensure Mix RPM requirements exist in modplant_ops; otherwise skip BFS
mix_rpms = set()
for _, row in reaction_rules_df.iterrows():
    if str(row.get('Reaction Type','')).strip() == 'Mix':
        param_str = str(row.get('Reaction Param','')).strip()
        if '/' in param_str:
            rpm_token = param_str.split('/',1)[0].strip().split()[0]
        else:
            rpm_token = param_str.split()[0] if param_str else None
        if rpm_token:
            mix_rpms.add(str(rpm_token))
available_rpms = {str(op[1]) for ops in modplant_ops.values() for op in ops if op[0] == 'Stirring'}
missing_mix_rpms = {rpm for rpm in mix_rpms if rpm not in available_rpms}
skip_bfs = bool(missing_mix_rpms)
if skip_bfs:
    print(f'Skipping BFS: missing Mix RPMs {sorted(missing_mix_rpms)} in all modplants')
    transition_list = []
    state_list = [start_state, end_state]
    state_index = {start_state: 0, end_state: 1}
    index_state = {0: start_state, 1: end_state}
    bfs_feasible = False
    operation_rows_fallback = [
        {'Step': 1, 'Operation': 'Unfeasible - missing RPM', 'Cost': 0.0, 'Duration (s)': 0.0},
        {'Step': 2, 'Operation': 'End', 'Cost': 0.0, 'Duration (s)': 0.0},
    ]
    tools.display_dataframe_to_user(name='Operation Sequence Table', dataframe=pd.DataFrame(operation_rows_fallback))
else:
    transition_list, state_list = run_bfs(
        start_state,
        modplant_interfaces,
        max_steps=None,   # set to None for unlimited search depth if you like
        num_threads=current_threads,         # change this to control the number of worker threads
        batch_size=current_batch_size,       # None -> default = num_threads
    )

# ========== Build state index mappings ==========

state_index = {state: idx for idx, state in enumerate(state_list)}
index_state = {idx: state for idx, state in enumerate(state_list)}

# ========== Convert transitions to DataFrame for inspection ==========

df = pd.DataFrame(
    [
        {
            "From_ID": fid,
            "From_State": str(fstate),
            "Operation": op,
            "Cost": cost,
            "Duration": duration,
            "To_ID": tid,
            "To_State": str(tstate),
        }
        for fid, fstate, op, tid, tstate, cost, duration in transition_list
    ]
)

tools.display_dataframe_to_user(name="Transition List", dataframe=df)

# ========== Check if at least one transition reaches end_state ==========

end_idx = state_index.get(end_state, -1)
has_end_transition = any(
    op.startswith("End,") and tid == end_idx
    for _, _, op, tid, _, _, _ in transition_list
)

bfs_feasible = has_end_transition and len(transition_list) > 0
operation_rows_fallback = None

if not bfs_feasible:
    operation_rows_fallback = [
        {"Step": 1, "Operation": "Unfeasible", "Cost": 0.0, "Duration (s)": 0.0},
        {"Step": 2, "Operation": "End", "Cost": 0.0, "Duration (s)": 0.0},
    ]
    tools.display_dataframe_to_user(name="Operation Sequence Table", dataframe=pd.DataFrame(operation_rows_fallback))
    print("BFS found no feasible solution; falling back to stub sequence.")


---

## Mathematical Optimization Model: ModPlant Operation Planner

This section defines the mathematical optimization model used to determine the most efficient sequence of operations and transitions through the ModPlant system.

### Model Purpose

The model selects a subset of valid transitions between system states, such that a feasible process path is constructed. This path begins from the initial state and terminates in the defined goal state where the target product is available.

### State Set

Let the set of all reachable system configurations be denoted as:

$$
\mathcal{S} = \{ s_0, s_1, \dots, s_n \}
$$

### Transition Set

Let:

$$
\mathcal{T} = \{ t_0, t_1, \dots, t_m \}
$$

be the set of all valid transitions between states.

### Decision Variables

For each transition $t_k \in \mathcal{T}$, we define a binary decision variable:

$$
x_k \in \{0, 1\}
$$

where:
- $x_k = 1$ means transition $t_k$ is included in the execution plan
- $x_k = 0$ means transition $t_k$ is excluded

## Transition Definition and Pyomo Model Setup

This section bridges the **state-transition graph** from BFS with a **Pyomo optimization model**, allowing formal reasoning over feasible process paths.


### Transition Definition

Each transition $t_k \in \mathcal{T}$ is represented as a tuple with five main components:

  * `trans_data[i] = (fid, tid, op, cost, duration)`

    * `fid`: source state ID
    * `tid`: destination state ID
    * `op`: operation string label
    * `cost`: transition cost
    * `duration`: transition duration


In [ ]:
# Pyomo model
if not bfs_feasible:
    model = None
    trans_data = {}
    print("BFS infeasible: skipping model/optimization; proceeding directly to export.")
else:
    from pyomo.environ import ConcreteModel, RangeSet, Var, Binary
    model = ConcreteModel(name="ModPlant_Operation_Planner")
    model.S = RangeSet(0, len(state_list) - 1)
    model.T = RangeSet(0, len(transition_list) - 1)
    model.x = Var(model.T, domain=Binary)

    trans_data = {
        i: (fid, tid, op, cost, duration)
        for i, (fid, fstate, op, tid, tstate, cost, duration) in enumerate(transition_list)
    }

## Objective: Maximize Total Profit

The optimization goal is to **maximize profit** by balancing:

* **Order revenue**: a fixed base profit $P\_0$
* **Process cost**: connection + operation costs per transition
* **Time penalty**: proportional to total execution time

### Formula

Let $\mathcal{T}$ be the set of transitions. For each $t_k \in \mathcal{T}$:

* $x_k \in {0,1}$: decision variable
* $c_{\text{con}}(t_k)$: ModPlant connect cost
* $c_{\text{op}}(t_k)$: operation cost
* $d_k$: duration
* $P_0$: base profit
* $\lambda$: per-second penalty

Then:

$$
\text{Profit} = Revenue - Costs = \left(P_0 + \lambda \cdot \sum_{t_k \in \mathcal{T}} x_k \cdot d_k \right) - \left(\sum_{t_k \in \mathcal{T}} x_k \cdot (c_{\text{con}}(t_k) + c_{\text{op}}(t_k)) \right)
$$

In [ ]:
from pyomo.environ import Objective, maximize

if model is not None:
    def total_profit(m):
        total_cost = sum(model.x[t] * trans_data[t][3] for t in model.T)
        total_duration = sum(model.x[t] * trans_data[t][4] for t in model.T)
        fixed_profit = order_profit_factor[0]
        delay_penalty = total_duration * order_profit_factor[1]
        return fixed_profit - total_cost + delay_penalty

    model.obj = Objective(rule=total_profit, sense=maximize)
else:
    print("Objective skipped because model is None (BFS infeasible).")

## Start Constraint: Single Outgoing Transition

This constraint ensures that the process starts with **exactly one transition** leaving the initial state.

### Meaning

Only one valid path may be chosen to begin the process from the start state.

### Formula

Let $s_0$ be the index of the initial state. Then:



## Flow Constraint: Path Continuity and Balance

This constraint ensures **flow conservation** across all states, forming a continuous and valid execution path from the start state to the goal state.

### Meaning

- The **start state** must have one outgoing transition and no incoming transition.
- The **goal state** must have one incoming transition and no outgoing transition.
- All **intermediate states** must have balanced flow: incoming equals outgoing.

### Formula

For each state $s \in \mathcal{S}$:

- If $s$ is the start state:
  $$
  \text{outflow}(s) - \text{inflow}(s) = \sum_{\substack{t \in \mathcal{T} \\ \text{src}(t) = s}} x_t - \sum_{\substack{t \in \mathcal{T} \\ \text{dst}(t) = s}} x_t = 1
  $$
- If $s$ is the goal state (i.e., contains only the goal product and is in the goal ModPlant):
  $$
  \text{inflow}(s) - \text{outflow}(s) = \sum_{\substack{t \in \mathcal{T} \\ \text{dst}(t) = s}} x_t - \sum_{\substack{t \in \mathcal{T} \\ \text{src}(t) = s}} x_t = 1
  $$
- Otherwise:
  $$
  \text{inflow}(s) - \text{outflow}(s) = \sum_{\substack{t \in \mathcal{T} \\ \text{dst}(t) = s}} x_t - \sum_{\substack{t \in \mathcal{T} \\ \text{src}(t) = s}} x_t = 0
  $$


In [ ]:
from collections import defaultdict

if model is not None:
    in_edges = defaultdict(list)
    out_edges = defaultdict(list)

    for t in model.T:
        from_state, to_state, *_ = trans_data[t]
        in_edges[to_state].append(t)
        out_edges[from_state].append(t)

    start_idx = state_index[start_state]
    end_idx = state_index[end_state]

    def flow_rule(m, s):
        inflow = sum(m.x[t] for t in in_edges[s])
        outflow = sum(m.x[t] for t in out_edges[s])
        if s == start_idx:
            return outflow - inflow == 1
        elif s == end_idx:
            return inflow - outflow == 1
        else:
            return inflow - outflow == 0

    model.flow = Constraint(model.S, rule=flow_rule)
else:
    print("Flow constraints skipped because model is None (BFS infeasible).")

## Solve and Display Results
Solves the Pyomo model and renders selected transition paths and tables.


In [ ]:
def solve_and_display(model):
    try:
        model.max_steps = Constraint(expr=sum(model.x[t] for t in model.T) <= 1000)
        solver_name = "glpk"
        solver = SolverFactory(solver_name)
        results = solver.solve(model, tee=True)
        if (results.solver.termination_condition == TerminationCondition.optimal and
            results.solver.status == 'ok'):
            df_result = pd.DataFrame([{"Optimal Objective Value": value(model.obj)}])
            tools.display_dataframe_to_user(name="Optimal Objective Value", dataframe=df_result)
        else:
            raise Exception("Problem is infeasible or unsolvable.")
    except Exception as e:
        raise Exception(f"Problem is infeasible or unsolvable: {str(e)}")

if model is not None:
    solve_and_display(model)
else:
    print("Skipping solver because BFS was infeasible; exporting corpus directly.")

---

## Operation Sequence Extraction

This code reconstructs the **optimized process path** from the Pyomo solution.

1. **Collect transitions** where $x_t = 1$.
2. **Rebuild path** step-by-step from the start state.
3. **Optionally reorder**: show all `Connect(...)` first.
4. **Build result table** with step number, operation, cost, and duration.
5. **Display** as an interactive Pandas DataFrame.

Result: a clear, ordered **operation sequence** with costs and times.


In [ ]:
from pyomo.environ import value
if not bfs_feasible:
    # direct fallback from BFS
    operation_rows = operation_rows_fallback
else:
    import pandas as pd
    import re
    
    # ==========================================
    # CONFIGURATION SWITCHES
    # ==========================================
    # 1. Merge "Variable Transfer setup" + "Resolve" -> "Dosing..."
    merge_setup_resolve = True
    
    # 2. Move "Connect(...)" to top
    separate_connect_first = True
    
    # 3. Merge consecutive Dosing steps
    merge_consecutive_dosing = True
    
    # 4. Split "End, Operation" -> "Operation" + "End"
    split_end_step = True
    # ==========================================
    
    
    # === Utility: force volumes to 1 decimal place ===
    def _reformat_all_volumes(op_str: str) -> str:
        # Regex: find numbers after a colon followed by 'litre'
        # Example ": 2.00 litre" -> ": 2.0 litre"
        def repl(m):
            try:
                val = float(m.group(1))
                return f": {val:.1f} litre"
            except:
                return m.group(0)
        
        return re.sub(r":\s*(\d+\.?\d*)\s*litre", repl, op_str)
    
    
    # Helper: scale the volume(s) inside the final "(...)" of a Dosing op_str
    def _scale_dosing_volume(op_str: str, factor: int) -> str:
        m = re.search(r"\(([^()]*)\)\s*$", op_str)
        if not m: return op_str
        inner = m.group(1)
        parts = inner.split(",")
        new_parts = []
        for p in parts:
            p = p.strip()
            if ":" not in p:
                new_parts.append(p)
                continue
            name, rest = p.split(":", 1)
            tokens = rest.strip().split()
            if not tokens:
                new_parts.append(p)
                continue
            try:
                qty = float(tokens[0])
            except ValueError:
                new_parts.append(p)
                continue
            new_qty = qty * factor
            
            # .1f keeps one decimal when merging results
            tokens[0] = f"{new_qty:.1f}"
            new_rest = " ".join(tokens)
            new_parts.append(f"{name.strip()}: {new_rest}")
        new_inner = ", ".join(new_parts)
        return op_str[:m.start()] + f"({new_inner})"
    
    # 1. Extract selected transitions
    result_path = []
    for t in model.T:
        if value(model.x[t]) > 0.5:
            result_path.append(trans_data[t])
    
    # 2. Reconstruct path
    path_by_flow = []
    if start_state in state_index:
        curr_idx = state_index[start_state]
        while True:
            found = False
            for entry in result_path:
                fid, tid, op_str, cost, duration = entry
                if fid == curr_idx:
                    path_by_flow.append((op_str, cost, duration))
                    curr_idx = tid
                    found = True
                    break
            if not found:
                break
    
    # ==============================================================================
    # Processor A: Merge Setup + Resolve
    # ==============================================================================
    if merge_setup_resolve:
        merged_transfer_path = []
        skip_next = False
        n = len(path_by_flow)
    
        for i in range(n):
            if skip_next:
                skip_next = False
                continue
    
            op_str, cost, dur = path_by_flow[i]
    
            if "Variable Transfer setup" in op_str and i + 1 < n:
                next_op, next_cost, next_dur = path_by_flow[i+1]
                if "Resolve(x=" in next_op:
                    mat_match = re.search(r"for\s+([A-Za-z0-9_]+):", op_str)
                    mat_name = mat_match.group(1) if mat_match else "Transfer"
    
                    m_setup = re.search(r"\(([^)]+)->([^)]+)\)", op_str)
                    src, dst = (m_setup.group(1), m_setup.group(2)) if m_setup else ("??", "??")
                    
                    m_x = re.search(r"x=([\d\.]+)", next_op)
                    val = float(m_x.group(1)) if m_x else 0.0
    
                    # Build description using .1f here
                    final_desc = f"Dosing: Open Valve of {src}_Out1 only, Draining({src}), Filling({dst}), ({mat_name}: {val:.1f} litre)"
    
                    merged_transfer_path.append((final_desc, next_cost, next_dur))
                    skip_next = True 
                    continue
    
            merged_transfer_path.append((op_str, cost, dur))
        
        path_by_flow = merged_transfer_path
    
    # ==============================================================================
    # Processor B: Reorder Connect
    # ==============================================================================
    if separate_connect_first:
        connect_ops = [x for x in path_by_flow if x[0].strip().startswith("Connect")]
        other_ops   = [x for x in path_by_flow if not x[0].strip().startswith("Connect")]
        path_by_flow = connect_ops + other_ops
    
    # ==============================================================================
    # Processor C: Merge Consecutive Dosing (Advanced Regex-based)
    # ==============================================================================
    if merge_consecutive_dosing:
        merged_path = []
        i = 0
        n = len(path_by_flow)

        # Regex to capture critical identity info: Source, Dest, Material Name, Volume
        # Matches pattern: ... Draining(HC10), Filling(HC30), (A: 1.0 litre)
        # Group 1: Source (e.g. HC10)
        # Group 2: Dest (e.g. HC30)
        # Group 3: Material Name (e.g. A)
        # Group 4: Volume Value (e.g. 1.0)
        pattern_dosing = re.compile(r"Draining\(([^)]+)\),\s*Filling\(([^)]+)\),.*\(([^:]+):\s*([\d\.]+)\s*litre\)")

        while i < n:
            op_str, cost, duration = path_by_flow[i]
            
            # 1. Check if it's a Dosing operation
            if not op_str.strip().startswith("Dosing:"):
                merged_path.append((op_str, cost, duration))
                i += 1
                continue

            # 2. Try to parse the Dosing details
            match = pattern_dosing.search(op_str)
            if not match:
                # If regex fails (unexpected format), keep as is
                merged_path.append((op_str, cost, duration))
                i += 1
                continue

            # Initialize accumulation variables
            src, dst, mat, vol_str = match.groups()
            current_vol = float(vol_str)
            total_cost = cost
            total_duration = duration

            # 3. Look ahead for consecutive identical operations
            j = i + 1
            while j < n:
                next_op, next_cost, next_dur = path_by_flow[j]
                
                # Stop if next op is not Dosing
                if not next_op.strip().startswith("Dosing:"):
                    break
                
                next_match = pattern_dosing.search(next_op)
                if not next_match:
                    break
                
                n_src, n_dst, n_mat, n_vol_str = next_match.groups()
                
                # CRITICAL: Merge only if Source, Dest, and Material are identical
                if (n_src == src) and (n_dst == dst) and (n_mat == mat):
                    current_vol += float(n_vol_str)
                    total_cost += next_cost
                    total_duration += next_dur
                    j += 1
                else:
                    break
            
            # 4. Finalize the merged block
            # Only add to path if total volume is effectively > 0
            if current_vol > 1e-6:
                # Reconstruct the string with the new summed volume
                # We replace the volume part in the original string with the new total
                # Using regex sub to ensure we target the specific "number litre" pattern
                new_op_str = re.sub(
                    r":\s*[\d\.]+\s*litre", 
                    f": {current_vol:.1f} litre", 
                    op_str, 
                    count=1
                )
                merged_path.append((new_op_str, total_cost, total_duration))
            
            # Move index to the next unprocessed item
            i = j

        path_by_flow = merged_path
    
    # ==============================================================================
    # Processor D: Split End
    # ==============================================================================
    if split_end_step:
        final_split_path = []
        for op_str, cost, dur in path_by_flow:
            if op_str.strip().startswith("End, "):
                cleaned_op = op_str.replace("End, ", "", 1).strip()
                final_split_path.append((cleaned_op, cost, dur))
                final_split_path.append(("End", 0.0, 0.0))
            else:
                final_split_path.append((op_str, cost, dur))
        path_by_flow = final_split_path
    
    # ==============================================================================
    # Processor E: Final Format (force all volumes to 1 decimal place)
    # ==============================================================================
    final_formatted_path = []
    for op_str, cost, dur in path_by_flow:
        # Call regex-based replace helper
        new_op_str = _reformat_all_volumes(op_str)
        final_formatted_path.append((new_op_str, cost, dur))
    path_by_flow = final_formatted_path
    
    
    # 4. Build Result
    operation_rows = []
    for idx, (op_str, cost, duration) in enumerate(path_by_flow):
        operation_rows.append({
            "Step": idx + 1,
            "Operation": op_str,
            "Cost": cost,
            "Duration (s)": duration,
        })
    
    # 5. Display
    df_result = pd.DataFrame(operation_rows)
    tools.display_dataframe_to_user(name="Operation Sequence Table", dataframe=df_result)

## State Transition Graph Visualization

An interactive directed graph is generated to visualize all valid state transitions and highlight the optimal path.

### Content

- **Nodes**: Represent states (current material set + current ModPlant)
- **Edges**: Represent transitions with operation labels

### Highlights

- Start state: green star
- Goal state: blue star
- Optimal path: red, thick edges
- Other transitions: gray, thin edges

### Output




In [ ]:

# === Export operation sequence to structured result corpus ===
import json
import re
from pathlib import Path
import pandas as pd

if use_original_modplants:
    random_seed = 0
corpus_path = Path(f'Result/Feasible/run-{random_seed}-operation_sequence_corpus.jsonl') if bfs_feasible else Path(f'Result/Unfeasible/run-{random_seed}-operation_sequence_corpus_fallback.jsonl')
corpus_path.parent.mkdir(parents=True, exist_ok=True)

# Collect static context
corpus_context = {
    'modplant_ops': modplant_ops,
    'modplant_interfaces': modplant_interfaces,
    'modplant_maximum_volume': modplant_maximum_volume,
    'modplant_resources': modplant_resources,
    'order': order,
    'reaction_rules': reaction_rules_df.to_dict(orient='records'),
}

# Patterns for parsing
_op_prefix_re = re.compile(r'^([A-Za-z]+)')
_qty_re = re.compile(r'([A-Za-z0-9_]+):\s*([0-9.]+)\s*litre')
_port_re = re.compile(r'Open Valve of ([A-Za-z0-9_]+)_Out\d+.*Filling\(([^)]+)\)')


def normalize_op_type(op_str: str) -> str:
    m = _op_prefix_re.match(op_str.strip())
    if m:
        return m.group(1)
    return op_str.strip().split()[0]


def parse_step(row):
    op_str = row['Operation']
    op_type = normalize_op_type(op_str)
    mats = [
        {'name': m, 'qty_l': float(q)}
        for m, q in _qty_re.findall(op_str)
    ]
    ports = {}
    pm = _port_re.search(op_str)
    if pm:
        ports = {'src': pm.group(1), 'dst': pm.group(2)}
    return {
        'role': 'step',
        'step_id': int(row['Step']),
        'op_type': op_type,
        'op_str': op_str,
        'materials': mats,
        'ports': ports,
        'cost': float(row['Cost']),
        'duration_s': float(row['Duration (s)']),
    }

trajectory_id = f"run-{random_seed}-{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"

records = []
records.append({'role': 'context', **corpus_context})
records.append({'role': 'metadata', 'trajectory_id': trajectory_id, 'bfs_feasible': bfs_feasible})

for row in operation_rows:
    records.append(parse_step(row))

with corpus_path.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f"Saved structured result corpus to {corpus_path} with {len(records)} records")
tools.display_dataframe_to_user(name='Result Corpus Preview', dataframe=pd.DataFrame(records))
